# Frozen original WHAM vs frozen iPhone candidate — 3DPW accuracy

This notebook does **no training**. It runs one locked comparison on the same 3DPW test population:

1. **Released WHAM:** stored official ViTPose/HMR2a observations, the official HMR2a initialization, flip averaging, and released WHAM weights.
2. **Frozen phone candidate:** YOLO26n-pose, released HMR2.0-S's 1024-D token plus its real first-frame SMPL initialization, no flip, and the same released WHAM weights.

The comparison uses all matching single-person tracks. This matches the current app, which follows one highest-confidence person and does not yet associate identities in multi-person scenes. Both rows are decoded with licensed SMPL assets only to calculate PA-MPJPE, MPJPE, PVE, and acceleration. No training, validation selection, or threshold tuning occurs.

Attach inputs:

- `3dpw-model`, containing raw `imageFiles`, `3dpw_test_vit.pth`, and the registered 3DPW data you used successfully before;
- the licensed `SMPL_NEUTRAL.pkl`, `SMPL_MALE.pkl`, and `SMPL_FEMALE.pkl` files, either inside `3dpw-model` or in one additional private Kaggle dataset;
- one small private `hmr2s` Kaggle dataset containing the official `hmr_vit-small_d3-a4x16-m128.zip` you downloaded manually from the authors. The notebook also accepts the extracted `last.ckpt`.

Do not attach BEDLAM, COCO, `3dpw-vit`, any FastViT checkpoint, or an old notebook output. Enable Internet and select a GPU. No Hugging Face token or `gdown` is needed. Temporary repositories and multi-gigabyte weights stay under `/tmp`; only the compact JSON/CSV report is saved to `/kaggle/working`.

In [ ]:
# Configuration and bounded input discovery. No WORK_DIR is used.
from pathlib import Path
import hashlib, os

KAGGLE_INPUT = Path('/kaggle/input')
SCRATCH_DIR = Path('/tmp/frozen_hmr2s_wham_accuracy')
OUTPUT_DIR = Path('/kaggle/working/frozen_hmr2s_wham_accuracy')
SEQUENCES = 0  # 0 = every matching single-person 3DPW test track.
FRAMES_PER_SEQUENCE = 0  # 0 = every frame.
POSE_BATCH_SIZE = 32
HMR2S_BATCH_SIZE = 32  # reduce to 16 only if CUDA runs out of memory.
SMPL_BATCH_SIZE = 256  # reduce to 128 only if metric decoding OOMs.

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

root_candidates = [
    Path('/kaggle/input/datasets/nguyntrunglong/3dpw-model'),
    Path('/kaggle/input/3dpw-model'),
]
THREEDPW_ROOT = next((p for p in root_candidates if p.is_dir()), None)
if THREEDPW_ROOT is None:
    # Search directory names only; never crawl the 53,000 image files.
    candidates = []
    for parent, directories, _ in os.walk(KAGGLE_INPUT):
        base = Path(parent)
        if base.name == '3dpw-model':
            candidates.append(base)
            directories[:] = []
        elif len(base.relative_to(KAGGLE_INPUT).parts) >= 4:
            directories[:] = []
    if len(candidates) != 1:
        raise FileNotFoundError(f'Attach exactly one 3dpw-model input; found {candidates}')
    THREEDPW_ROOT = candidates[0]
parsed_candidates = [
    THREEDPW_ROOT / '3dpw_test_vit.pth',
    THREEDPW_ROOT / 'imageFiles' / '3dpw_test_vit.pth',
]
PARSED_3DPW = next((p for p in parsed_candidates if p.is_file()), None)
if PARSED_3DPW is None:
    bounded = list(THREEDPW_ROOT.glob('*/3dpw_test_vit.pth'))
    if len(bounded) != 1:
        raise FileNotFoundError(
            f'3dpw_test_vit.pth is missing directly below {THREEDPW_ROOT}'
        )
    PARSED_3DPW = bounded[0]
print({'3dpw_root': str(THREEDPW_ROOT), 'parsed_test': str(PARSED_3DPW), 'output': str(OUTPUT_DIR), 'training': False})


In [ ]:
# Runtime-only dependencies. Kaggle's existing CUDA PyTorch is retained.
%pip install -q timm==0.6.13 ultralytics==8.4.146 einops==0.8.1 yacs==0.1.8 joblib==1.5.2 loguru==0.7.3 smplx==0.1.28 opencv-python-headless==4.10.0.84 scikit-image==0.25.2 tqdm==4.67.1 progress==1.6
%pip install -q --no-build-isolation chumpy==0.70


In [ ]:
# Materialize the reviewed, checksum-verified evaluation sources.
import base64, gzip

SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
embedded = {
    'evaluate_frozen_hmr2s_wham.py': ('7a1639d8fb115041daa41a2b93b94e03a0937e6e99a82e630222303fe6bb8527', 'H4sIAGH1p2oC/7U9a5PbNpLf9Su4vA9HxRRnNBM7OWWVOmednL0bP8p27daWTsWiRGjEDEVqSWpmZN/89+tuvEFQM3ayqYpN4dFoNBr9QgP+jz+dHdrmbFVUZ6y6CfbHbltXl6MwDH9p6k+sioNsvy9ZO+nqCf8KLl+8+0fAbrLykHVFXQX1JvjHy+evg6zKg27Lgj1AYMEafhZ51rFkNPq4ZQ0LMvif3WXrrjwG3W0dNPVtOxuNpknQsJJlLcs5nENbVFdB0bVB29UNlP69+PiubtnZy9fvL7KgqPYHqMPRNmWxD7Ib1mRX7IfRRRL88+2vby+eVZM9NA+eaLjYMzmffIAyxLDNdswZ9JYVV9uuBWTf1N2WEGiDrsmKiuVxsCm6Dv9uoc+avuoGZlgWqwZmmCdB8BP0ohnxaXLqsHwE9MERi5xVXQE9ApxcySZ71rRQR7SEYdbXfEYNWx+aBtoGmwaQbAHyh9fvfkVkDi2BA+Jlm441QIcNULVas6CrEZf1oYQR+QpkAP4/WyjdAW0mMFNYqRsWrOr8GOxY1xRrmCms8Wi0aepdkKabQ3doWJoGxW5fNx3gUtUdLW87Gsmy5mqfNS2Tv9ftjfzcZu0WiCF//gZTk9/tYbVv6jVrW1VyVJ9dsWMchXVdImlxQInDX+pDBRONg5xtskPZ5cW64433WYfDyYbv4Cev6I57Wjpe/rw6KuTlisBMyzLdF3tWwtKmQPqc1ZtNkLVAe0HQFBv3Ou7qVVEy3fUy399iN1neMGeequftNtulG5YRiYEebVd0B9o60J0qzQF/q1cGKavDbn/EdtVeEa1u1mK+211z0aYb2qly1tEogP+Q4z+kf3n581/+9u7tqzcf0w8vn188fRablW9fv3710Sx5//O7tx9efXz7/p+8lIsAquMF6y1bX+/rourSdrcv09VhAzRr49GY4/Pu1a8SjVc72JS8FJveJeVKrSx8ivXCqcjSqhKF/8p3sgy/eSkwQJOVR9hECgzu9tHo78/fv3r+5uOHYB5EodzURPMwDkJOm5RkUqpkUjgevbx89jr9+Db96/Rb6Ll4FgdP4+DbOJjGwUUcXMIHFE2hbIqFUDqF4imUfw9/nS9HoxGwZdBuM6BqhAw5Iz4cB5MfQW41M6JXXlyxtgP4YoMkov2Yam8LkBnYNan3rALkV+EYlxq6s2zHIeB/GxA2q7JeX8OeB7nImqjMdqs8m4mWCfyRR98H3wBiF9+Kv8ZxsArDsYai8UkOeyRCRDA5Kg0D5qxk/Zbd8S9AVMzzWK23TV0Vn1iUs5tizWZ88RL+i6b9BmjMxys2AS9PYEeyYD4PwvUhz0KNDe+MhYkJW463rqs1oFjh9mkPu13WHIHITdfOgrJou0W1T6o8a5rsuKShUTYsgBogqss664L/A1J1S4UMCLOAd1cIgGgHHfEeRAzIoJ+bpm6i8E0d4J4FHXdLCmu9PuxIqua0BpmQnaFFM7WBE44oTgPQM2bAMR/j3NZl1rZiX6Fgf1UVXQF65BMsalUlr+v8UDKxaCCf37ADcr3WX6QMYL6os0DAN3XdTdagLlBNbooGeG36HUgQmHybkHynVQeCpmkBQ6VppAgAumwTq19iJ88MSvIl+siqtm6WuiXNl4ZIG3bVgMCrm5nVmLd1WIKGPIBeisaJQmZs8XgF6irmK4CcLlBKgON3beRwMmIPfH8FzMAaIYciA8B4dLJl6JsFyAtfcUIsFeH6SWoCtrdZkzvEDNDsSJ/lDjWCFeuy1i7UFDJL9RR3GTIaMOLc4K9GqGQYIu3qlNrcRRZZBAYwX5A0exZNUJyBAHum6aG/wHDqcJA4SGEcEMo2LELbgjQ9H8dWE4lm3F+bmxSWbY+bx1NJIPOi8XXEKQxU/VUviq9jhoaTrx9MLRUGXtwj1wWQdf5LVrYGoppGfC8Befg6saKCXR6Fv93Eq5v15MfVb+sw5oP4WCdWNPZA5B+LWRzMpt8tVT2YFzeFXb8g3TNdLpMdy6ooL3ZzWI1rxvb4+bEx2V1IJTHKRIATYnVX56xMuWhOwZ477IBiwwIdQIAkVdpMgD4fFO8BAyrC9/4Akmf03xyeslNTHB22PG0gUy3Xq5Y1N9za5CxIbEzIzkjL85Uha2fWt0oKNDVSVKRSPaAmFiKLQK2ybr1NWxCzM5yUAuev8NEDTBxbz4BxuVRy+jl4RkftWuSsA1sWJFq3xe1Tl+g6/OuAnsG6qfcxmfpV3exQ8mfSEizeftAy+y4l96in7dBW4byyy9rrBxvhcLKBKWlgWVE0my3raoNeCvCqaE9iz2hBlGwZNMyRPc+Tc44qiu62AxWHQhsNtshQstUVi87joAT7xlimceyuiyFYctau5yF35QKTN0LdBqy8G+bu2rxos1XJ5qjtwclI2i5nTZNAadcdIzGCoUfqprgqKoAhJkwma0J/GtPmKPEFTWkOD7dHkuBEyWDT015wMs0EuZ64RFjaOo7sQz4IGYh7si/RQKwPzZrZjc0JATK8CVohKIGi8P3//BSOBzu0Cfj2rMojWQAd90dUeW4PhxKyX6+dCR11CFpF0bNvgRXgD1h+Pq33DPbLHtypq+SnV7++evPz8/f9IYXeJWmOdANTZ06eYwLWxCZdczcxMkVgC94C8qiWIwn4Z7h5bUyd2djKodhdtZ/miLNVjBtlDsx/Po0dWCgn5qdErKM9t1m5mXsFqd0OVnAFExlWUn3rXNdZu/aJl3CgJQRhRxYHy/WLBUWRmz8Ve8Ukraxoxy7j5t02DrakcmEZFCMgGzg2RtuBMZizO2g2mfYIzSWSkDbW4mzE2MmqvgM7CTQmbnsUaiRdUeCYDcbBj8F5f8foMVJh+c8tuLiBNqACumy9BcNVWIIJaDj4k3zzqM+w1qRAq6A7AJywy+6i3nieLWbNm4/Y67bQYyyN/SEoY0D4M5LuKYarxLSu2XHPbQMgGZELvUuN8Z99ZOLaSG52mA74K3UbXX4XBzly7hyKCNPLC8+ElKYyAMDAbTR9XH9SYrIvonyKZqqhLvI274rqwGzKSdKk4B1nGE7o9XNpmGBLcy2+hFXG3sGRAy1EyEC8WIJbj2zUkobvY7agPRecoXhNztXeE7+XfTo/ApXU4sQeUhdLW0qBMZOb7YzePyIXXto7v75L7453x1N05lsQW30tjfG/04R7mHhx8ABtvTBdescevTb8CzYxCg6iKcYVQBUjEb/rb80duNK7w84gfLugbuAsFFWU3RXt/Ny3J+8Gu4GcGuqGsQbwou9i+UXrF0kknki4Y6DORXI+TroaTSXPqrTAGtAXB+PLCKTCHxKxiZwZzPybYJqgB5Q4GKG7MRvEUUnPSPLa4nwJKKpfF0uJ5hCMowfG1IJxOQzDmKGXQzhgb5VYoeFKcwfBPIBaxhzjx3W7tLpNT3Qbjx5fimt1/tQPC2riU+YdF811kxcYNkNJCIQFiH06aGk5MXf3os+fHtHnbDxYP1wqq7CsKxG55auEi2FZNY6aKus19+Pmp6W0lyx8koptz9ToEyV5REn8cP+j1V+JqpMAiFNPtPGwxmnxZhPHth56oJxgaWRwgIo+XWI8WxJ5PAYK4/jR4JqOTtofkU9H/RlVlA+yDQyNEdzV7qlPwh38FOujQUcr9kjQmOiPf35i84unz07MxLKE+ihwV7Or046c/Aibu+ifNpREKAAcA4wIgg1NI3IQXX0NYFUojJuA1DamEwe50Xi1tS7ohAz0pOjkia4UhRwa9EQ/fvY50PPyREfSgmhqiPjrgtvF6BDRV8xZAHwhBhYHwwNgQSUyvalOOyNLFWqyIydfEjixMIJ9YAeuepETeVjwR4dLNEkshHpBDRs9bSRSmVoPaL2+BoGNlOMkXmoaU9yEg1+iDRF5PO0YaFyldIZVVFcUBf1C1/hx0QQ+FWL+WPxAdpbfxJ/yB2c6PO/DUGVEhY/DxeSQB710fYiGO3IhKQXjGsja1rEVD/B0weKBHjTDfg9+PODvIjZfvxOvcHo58ceUlKetSQ2xNaTPOVPKvj1wP6IfbIaxPytsw7tQBnvxqDnljgTA5zzK1dfY2GchapSTfZTKAV21qusyMnvT8lB3XD6jAhcBymmJjGKiNJRzVtPlnJhQIchtVBExfBgaVPKpOvu4x0Sa56PgYCRCdQWPoVF8dqadFjEEd118rQ3VC/18h6juEppgxFaBnp/DY13WF88E3eQmioOQ9iAUG1vrnoO4f+B0ojmojIG6YVz4VKy7rZvrWaCOaTmsO9/ZJy6/r1wkgbQn6lIin68BnZle7wercPqDlXhS7KsE1klB19zgSYtd65x3WKfB6uDj/YHnN4kUFEqloiwjin5TABur2R0mbICo3RVtO1HHIyu2zW6KutHHHsKSIaEsKJ5oAye6S9ZgpMIyxURiYVw0LAe6XOZQWEvW6thdZ8AQFbDQsMpNpIaJJVUNUEiqmH8CWQwgXZP9Rogf05xxQPrA1ho51kSPDRKPtG6CjcKuwHbA2DMmtCm9uEZn04Um+QbmjSd+k6kNxCKXhpyU2ZE108gd63GdG1YedF+Wf8GQF1/cDQe7+LrBLvvdyAy2Ohs/njhrNTKDV9YmBGbLdvuIYj4q9rA5cO7kPb7pD/YkiKAxaGtqgcELz3jEXdyMoE8SuuIb+EV+QY9s7eHjHvsRVrGWAwan9dWdVDMKB12F20hW0ZbSVcjMsgq/MQQ4nS1NpcPAGCu6o2wFv02RjVOUVXy+HlWmKGDUcSqoSv5TS3KU2NlNVpRozaZcT0Xwg5VWOgse2MbcvCQ9JU+4rbNtK+iC8TPbywdjnANegAe5FGYrWmb2yQXasdAArdi+Kwg0vsjDvkcdrlb1na9cbn5vXVmAE4dr5Ycp6wdhi/pTYxC7+PAls8QHFJcgLVyEjBQUXSECQirPTGp/2IFNkVWdSAgwUrHMNdVf7hH4UjAmGNLgHbT+TvyEe3nibJ8Hol02gg35+V65cAJTXG2ZizhzziIXog31tIgSinRctGT6kRGg42w4F64fgNNZVNQAUTIop7CQKVVW/3t7tUKUN6m2swQhFQzd+t7cQHy+Mt0kgx3k5IFRCnEDdJDpxMlzcUj6jmoi8whyB64oa1LKmpv3OrzgecHtS1buf5GNDcnHh0qyPNcHseFkQsX5BPN3QzzB/NehALlCDmQckGtBiZynQGBeB5sAgAlJxa+EgkbvBIyb+ncB0Dm5XwuGDOXfhQiH8Psx4ab8RORqfS0UzDee0Dn8JIfOZLF99cwun+0mlEw1MfMEv365/iBYLXQidzgUPTB7SWbKy5Obgb7ClfvyjqgJJuTITzC044WgQn6DnPI7IdDqPgDi4umzkzDqQwd28NfSfs8aRX8NyukLHSgDhYOgvxBIK+MdcmTr/BMb8LY53S+I7RrykvUuc2r5ZYDBar63+nmQvC+weS950TP4I5qgHAnOgrAsVme0CdszLE/2x9CLsGz/7YvJy8Muq9ozLJddV9n6eoXJAmc3RadhcBqjN4n3PeYB6maRH2XmXikiqxzwDlz+Nt0UJTiRS5keLuBorS1WjfG08NdiGAUNu4NJEYKtj3cMEqRKJICIeHuh7gZ412Yc/Gk+dEfjdI76R3CnVTq4BihjznT9p6gqwLLebIp1kZXyppPI/+JMVO92BToW+npOQsBSzs+aJRfhVYHsHTbshqtO/PHy5+cvQjCn17f53F562Ajg5Ohw7DiBpSn2wtgQdPijR9ecdHp4WBVz+n8ys6sxdiHuxJxcAdvSD3++29OVMB77+OyFdw+mZn2oclHNR78PnXA1IGeRR3PIw0gZeCjm+Gz21iiYY9yHKqkdw9EqCMF/RiKrFxAzrmvA9lHeFhh3OuXXdL7RLLJy1+nsjqW6lu8LMqRSkHRUhtHHurxhMvFw32A8cRO+EAm5nzle9z8ACW4DmYv5WQO9D9GoP7RbIymaO2yUT40XqwCTLI9cQTu2ZHKq+hhe843l0PicnNB2oFxHz3XwfI7dkEM35MidcuBCEdq6zHuFjkunoHh7WJVuT/e36xSGVwyc5CaU7joJ/1XL6ICxpaBg5JJ9ErSsE462Fqe81wO7gHyInF+rRCmNYlFqiU3Byhz5hUNSvL9DW0KoEfOwi91Z2ZDi8A84LwVWYLV9AijCAsQlS+1YAd6RsSPOMJk+UiDGSdPuywKMijTEcPvifDnG/QVWqxBYS30gi+workRGCMUacVEsPcA43oionCOHyu+gpuIO6iOmbYIYWYmGiNjCgw6PjPRRWmKS69SYG64sbkZl0OJJjUrhtDDFHE677cw81ir47VE6NyqLqt1na2ZLa3Ge6gCd4J0/G65xxgQS6Nm3voNFcRWYjA8T4AJllkRnzNdA/kQ6Kkz5/O0cJgOqBdS8yCbbPHiXbeiqMV9Kxm9rK0kaYF4fbLzQlL2aFcJ3Td3V67qECewxngBTgL1E9BQIje+dIfnF5h8Aa36VGjcZnnkzcdXXkBNaaquAkHHq4qiSjN/vNE5n+iao1iWx33gWOs8wJ0kzwlDGBZNoyGLtgXctPGF9dLU86k0Q+Wjs3G+B4fCmA+1oj5kuxZ++Jqjwc68P6sNX/yXdAUM0dq5k2qeZWln6zP+xSsGMT06X0OAGPeVXm/es+XIaLRyS65pUOfMm8furCK6MMmSM2YyMfCM9LZ/jM14Yl4OXYmbyyPpQtSAk2CcWnZvTHX1tmNIyMkRwbWZeeY+wz3go0mio1ceEOh8ejRo+Zjjj6pEdHTXuwrTXZDr0msEfZjN9hIzZMbwn3V8R0XwqHfU0kp0yo95mcNJhSO755FL49akwpL1TOjEa1H1atMk5DBxSiEMJy2gg3hQdexcXFEA8oOA/YrOHBUk0xnsAzg1avjYyC+xzKLVfOFPzI8crA5JBWQhS3H2eIrzvZZfxvHoLVQBkX2TtarEnJeGUMd2nHl2csnpLPFPjZlXk3NyRFldsTEXhsphJqjwJpkuPbjcvneGh4OnLitZNG5JUzjUfFLnOvVPE2S7iHol9H8zTwk6u8t1A8l4QsnbTE34rxZzJQuZ3GJQ3dqW3h5H4YVu8djOeirJcgPF1zk8ewGvEZMunJ/mxd7bxKAYVG5wsLa0yaW7h/YnESulpfkAk0Ez5LEe4n6H5QnfreeBU08XjbA5fMCEI5D7hhSiLQuRDLRez6dJUKHZHmQZp9+Te1mBX5c7RCbg2FTQusQleZ/9M6R2Mp9MeKCtNwZpxpAaLHSTvegg641x+N3ZuxvEkBx8n800o3gDqHaZhIld/8OlsOaS2nTMvkddlg6DCL4GifHEXEk/8+gpQqczn8m+tx8MTi0SZWvTlqxf5AZpLnPXiDwr4eoocAd0TkwRg4wN+Nohed50ko7LWeMbuVAjulqcJ80uWvcnd/4EJn0LOUyQSzVU7GUx4JHHwzTcmLz4uy1Meog49S7Nc8IcpMIWKhguXKHvtnfa43FDf0xoywRRsJ5uR5DHv0v/YxsNoI3xCVUDSco9fc9c71rH+ZZBaNHDyLUz7SPNBGFpmvzNbFa36AweUQE8N/PsZT5BKc56NOTKizYIWbfskeDwgm2jmTWrxnNIQKPH4WioBiFvWwFVR33bx+ubWpGMH9y8ks+JT+xmo3q4CSZMTyr9ne+lRYXJMqfZhv1MdvvhtYGEFCPt3MQ0sK1Um6/puz5y+0Tl2EbVvVHwFptLq+DeiyqPGjnslQsk6uljWt9YGUp6BSJMZzttxcnd4vh1n3H7WVv9ZMWdfOMosGpKTsaVbjF6OvBU5QShsMM/PuDribD8eHOGt+/TWc+onaRlMO1jp5Jvba9MvN2I4/UqMs3hukTkeSz/yM+TrjL0KSiYngUbTlBHZknyv2xS1gDT1rY9h+vltYhR0QwT79Nv4fJWBTDm0E4WmsROy3BvBMh6Dz0QMcFj/NizMynbVek6EWflA97QhdX8SBF4kFI+VOBlquOH4ksTynTU8VxDvUXotDyeopheY91pKb1E8puBFXzRVdzl5W/6U0pgubYlLVgiBn2diuprLG+q+HHyPnTembLFjZvtB45HtZFpd8aXMJAfp6tm+n713NR/HWQ9zl3FoKWTZPkt3+9/2LN3toE9vUj2lOnhdODQheVsN3DUOOTs/gMjwuAObwo/CwkJzCCOH9WePZfw+OCf8YDdwjz8ME0Oc+vDXch848THOeNY1SECGRucKn6YVFzTk66VtKC9N8SxbysXopd4aOy+WMlaelIvHc+cKhMfqkrmt4v0oehfY7DDsBdk9cS4ZHhCLYitnylpG846Wp2x/45bAFBlYn9IpuDzf7Fsrxyhbgal66AC/Ld7JtDQCR2fGZyblDCKfgZcPRqMkiFs16ktElITuNM1gu3jC98uxOKOc9iFMwBlnk/+iM1DjgZ1Ho6WYLNXHt/zesIL1mTSIElr8PBTKKD0L30vGl3ywidSqdELs8NK9dZ+BHn+1cjPa9ZbtshSckbagWKAROwrZHVjzBSbvYZBQ8JwFP72Rj+j2TTbj2p44wsQjWcxNZhiQce7UhntxStrL5w7l8SrgwLcpazsnQz7UR6vYLCtLnZzgO8HgF6wycRfFiPCJSKV4Q80IWI4cVcKPTFIODgy+DXpHWuZdHbIGZ2md8rqRG80DGF/XUxBQ6cpgj00cIOpgQcSokcaptrMU++pdjTYevh2GMeteFI+eEDFBESoAyDgl6WFgMhxQXzw+rrLo7FfIrSMCzFZQ1KeyH+y3yf2PjbvLMWTTBaH5rvkPnmfN8fr95AW/UMpfRjfi1fwBcQc9L0J0Uk9+scso9soCSraqkhcGyjy9ajDNzLc3jIs+YNBfVRlukhCjfIDyFYAGd0zcIIo134uXbmnLyM31A7o2wDJ1k8tkny0+kV0LHXN1bOp2Xe+ZeCDZnY7tF8xkkhLAKsEdqZAsRDSkJB4FB/hItPGop/n4fVUeXfCoMKv1Mb3NGpQXOIB5mQJzklYiA+Nv2RXsanwnsj5cbTHEQrzEczmLdzQbAS7sMewGFHmTQkuagyCUvDOmLysGomXAdQdIDVwoo0GsMkfdB+OR384obkvPzwcreuQSLI9NfZAqwmTje1sS3rAq45aqIwtVJKgtKDkf0N923b6dnZ1dgUg7rBIQGWfHGhBugQvOcG4JZYJ64PBURnmJmP/ytjPyIig1F41LI0fXaeTKOZ37oZD2v5ru9FDomT/9LR9A8ETuhgEFdDWsIl2bwfgVwjjsPMEB2QzfzWJlJBOnRSmqZ4KXqCL3wpA7vJ254p+Ck93iqj+dkOnvbmZsupPHRA61RwcI6Ev28LGK0+YEvwxDu/fdeveYBeb1ef3DFcDmAW5aVOvykKMpIg5Ntaq1GroSg2sT41yXG4d2H+vavzraEYYFHwhcsTUmbUufyKPfLQh8GNuI6A/Tnwsv8JNUehE6CmO9pEA6FByWQ6tNvb4Wc4z7HgDH7EY78QQ0p7WV8ooMw8N+4p3qZHeNaZ7i0Wpx/4TdFW2X1tfGibLZ87YpAFdMcI+MwAE3iPmhRdXNL8Z4LeF/8XSarrvj4yjhodtMvhc+37q9oaQJfgevxVi3WjcZ6MfX7s2BQSjDooMpdBeFCQAQoHhKrZVBqux467UGYf3335PwPyThfTCicezxf5fft9T/OIMklfgHGvC+YMVu0YGehz76+v71Blo0DGUDsORFse7+QQURbxdzGuL5WTvn5EQ2gD3QZjTteQimEp4Ejh2InBu2LMutGLhZif5VRK+Q+FI6h2JPnojnY1wfj0PweAfAMsG9d2FNT3fIm32M43riyqve+A9i4Pe1/6jhHcF1SlAZxEMpENK/CRIZe3fsG8TY8aKH5HSnuYOiFDGx7yq3J4V3VOC/PIHMnabk1qcp3glOUxGo5heER/8Pb9Nk3IVqAAA='),
    'hmr2s_frozen.py': ('4e5c70e17373ed5081d82cffcf2b18c4ac4f4edb27d027a39b7e9bb7bd1811c3', 'H4sIAGH1p2oC/+1aW2/bOBZ+96/gah5WztiKLbtuYsALFE3SdCaZZJNMdxdBINASFRHRrSSVxC363/fwoqvlTKaYHSywawSxRB4enuvHQ9I//GW/4Gx/TdN9kj6ifCOiLJ0NLMs6YdkXkqLT8yvXmYyvEU1DwkjqE/TEcJ4ThgpOArTeoJ/x/X1MEE4D9D5jBJ2fIfKcZ0w4g8FNBPSE3keCKwLM/IgK4osCCP0sIShkWYIEkOEC5mb8r+iGFal/Sgn7dHKOGIkJ5sRB6CaifJBkQQFz0VSQVNAsxXG8QeuCxgFHWQrPktMnenOZcQJCr7H/sM5SLdz1+eUZEgynPMxYQtggIjgYIfyY0YCm92oodNMUXkYwcRoQph4Dyn1GE5pikbGR4pXlevZxOcMgILkckfqUcJBPcWOEE6kxPOQZpzB640jbDgZKa88LC2kIz0M0kQYD1mkmsGTNBwPTpr9iunYKQWM9MscigpZy2CW86g6xyaUqpv1duqnYwOS+ISI0zXJeEjEQEYxyTwwHSVf2pWmj0UnTsj0ED2kDIMzRyWAgw+Tauzq+vLj+eHNx9S+0QlYkRM6X+/v3VETF2gFv76dCyK+mhx3otsz49xfn5x9v5NhgcehOD8L5dEomk9nCnYbuYRBOD+eTtTtbT94uJou3IQ6m5cgPFxcfzo69o6uPn469j0eSxfRh8RD89M+/i+RCnIbhB5ywq4LeT59np19uHp/iqJr19Pj9z5cXH3+58a5P37lvFjDaHiD4WAfu7K17QA7mC/9wMvXfviHBeuoG88VhiN35hChB4MkPg/ncdycH7nyOD+frtbsI/bk7swZDM8n5xdHxmXf08er4fWWfhHmPVIx5AlHsBbMxnj9PF+Nk6h5AhEBAhciLMxx4WRhSn+JYUns6A+w6opbK/UM0/pv091IJzrOCQaaWeshPPaBq2kfW/Gh8WiSQElazFSRzWw0wKYnbNGXgt1tBQiff6Kah+k9DBDFtJHIo90IK4g+XtWCYcoJOoPWXTJxkRRocM5YxO7QujN41CEFml7pRjhLKOUT7En3Vbd8sPSXPiQ+6txPHka2ejGYlAVjWV4lmW7V9pd5cWtkamWkqHRRPmPMXiSYZU++O9A4AoWnuqnRVAEQlpNTmfVbEgbKFSaKO1AbatuTW7VpyOast/9WKGiEc8gzqmeDQX5qGEYCY1HCHsPJjzDm6ZGBtlthp6pxrYi29CjoPIFB4ns1JHEr0S5YScEcAeoQpmy1RPU7GXVt7XgCdPXQqPsO6Czg6KUwMagKLM7whTMkBk3SowhRoqhkHlXSA3U+YBUa4RxwXZGnw6YakXAL03t4DkNzzdruStNnQ8Je2kZnXrqS0FfthzXFYGfCEkODEiPK7jBjRAFYJr2z4HvsRoc13TT4XchnEcZ3m8iMtS1OAdWnW5ozDUZfuw/HZr/Z28xHL8qwQ9sSZbHca3jVbpd5rmQx/hytf6TIwiPFU5Z5raH8nTJHwOx0EVQE3z9DqyffvcBVNU8KkdcBXiiXaq9i1/al7DVW7i/tYQUI5cG9vPHHetGmwVDMwEZGFIsHP0u+r8bQTN4H2h6ZsOqdNJjLv88OjSdBGGNUK7aHZCK0p5qsTHHOyPb6apRGhNbeKkYmbrjh/UITAzGwzQg9kI4s5XsRSpoaGJmQcPyrSB3umhGkZrZdBO9Oq2skWBnusNUqRHaFASbZGEbwGsJxEq9rXwxYP0BHp0bJstLdnHTZyp3zCZWyXOukoaEun7ZJgkRRxg6+jamCoBog9no7Q2B0OwaV1vPVMB67JlU9bLJtRZVcSDbflbiarDg+7tpzmrSynjKXtZoxoDeucfs8yzl+Z1K2QbGa3n8HW4Vl4r8h3XcT8zyR9J+cbhmrnvvty7n9+ATn+ZNToBEFvrWDUbDfXnu+Hln5MAUgx3EpQcXeAShOIypXr/6jz34k6N/WBwfcDUPVWwg4oNZ2480YHyUVUds3q9hqdoH3eYlQjleS2qPuSOPeaE7kHdV8X/+RsB4bvd9SisazguU5bbYwzyoV9e1fTyUjzZJBpwytFG9u/DitHni11Q8oUlI0Jtrrl57a3VX7K7Y6Cj3ZtqEtkaeUa/oedYraP1U4C453RiwSdUFJCtBC3K9Fudq+QVXFs7lhUgwmTXcrebbW2EeTPBFwZQ5KnV+U7MJImbDaEoKBnRJHx1oiqdrgpuQyE1QwMEKMfdX/viM6cekil06qE/xd5NMXcNaeBLt3Yg0RHxM9g0//bG5vvSGmASpE9wM6OJGsSqJPR5ooO2C2RqzMIcL1Lf4kZToiAubRPQfogVcMNh++sVGorAOludP7t8FRa/iHh2QyoamnaYUxbvTeULwfr7x977Nm/otWa7wrDejd8fnm2HT2ngCv/mQja5aIybLueJ76sC9phJldHNJ3Pt2l5hPNe4sk2rY+THspZh5CRe1hUoLxcF2EI8llKr3UWbDwpmDUyfv9CIMZUCM+3wncHDyIw7xk/ed1wEH978ExVJ9JR8rzSY5lYBBBlHtRHjD7bzNwgvGqjbFnWr2B5eVlBnrEvIIbiIknHa8wplwH1aAq+6kZDXccE9ZGsXJ0cdaehItRMDkYvH0E15TFV9rlSfAfCJykEsSeqYQrlMkQsvS+ygpvYCCnjEgpO1DkcjukXUil2uxwh+JvctatrDsGVBh7DT43JDe30rkHSYdvwQ8VgjGwtwV6jdejwIlFaPBCSy6lvmATvPS1tqzYDqUaNQ3ARURZUhaxaRfQEI8O/rYrJc03NBfYf7C654jish1XnkuraTt05bOU2eOlqy38hA9sjuZl8oiJSTv7H6btz6fuAShOqqzGJWepWD8bW3u45R+veSwAqRcR/yDOoNRs3Fa8GlPrKA8z3muuQTl5VN4CrBivnE71pF3A0ufc4RMPKdt8sIF4O3U5VlGPhR5qkWWrLj8JpWUmtqkq6VdWvpm7nbLTQRR3f6lGH3atpu7HgxKuNqHfPbQpZyemhnfk/Pzx6asstI7UjGixTnrxHlCPJauJ0ZoUlzIPQuydNnh3j8iSPlSZg3RfWGLMUazNupA+rVJAvdq0cFKU4r65mVpafF4B/5gLZkxe8W4cHkOfSuV9bwsN+c9lTgskyUm1x9VoLmGbkubUUH3CiL6w7hwqScLtd7tJQ7WKBjgkuU8W2qzswB4S0KmM4VgPcv9W7Mn1bNUJFColEfEGCsmRQYV1LYKtHyHXAc19olBk05DCc5DVUzWvZCaStG6itcj60KhCoPSCZJzLWl+U0q6/m4Rso2cOkFmH1tX7+Zg36tw5KYQIOeEWFVoE0WOdzgRnkAUTtztqsyGNy2y7nXvt2V5vvByR/tJA17x5RiBMabwDcfEIfCUcYAUw8y6tiLZfcGOTqhwHytwKgCW+wm7kop88klr9QQAT7EYoyRr8A7AJ7TgOi4pIKrpgC9KhrTprmhXC29u2QH2XUlNFn95vJLH3wN3OX45nb2JIbZnKdrA5EGhOoUxEfRehJn4rY8DREvlUz0OvBqlWZtCGuVyRHFQK3sHK3iKcvvgZik5PVDoaqs4u5j9TfPUD19iFaqVMb2Vp1tiKp6myoBz4XhEAJMW0gHBQKnqlmO5xMmWs2AGWlX/e3i86KoSohe7npsupldnJw7XfYkDHcywu6XuYEBINGUKsaQXZQZWRe/45HF07iKYMi7AliPlQp9Fd4YvLHRZVbGsyqqlGVsD7hch0ADFI/ZRG8nYwY5bDaswzIFArKNazOk5KDPFrvrY6Ne4btslRuClr7gqcIJ4rQW0iYLtmWSQVl6/Jui8Vi6/DRREyT20h7dGS8YUr5GoE9ZXi9DeB2b/UkF4lbWB16MewPXWLL5bV3nVQUKWjBW0fUljJKAGFg1XlmPXqwqOYxjG62/uTBxgfMCOI3m1Wd2WGQQw6notUUr7lnJLea5XZ1bv5VSrfUWtyGaol2VNs3WOUDcIMfQdkZgmrCHioYlp2yMFBqfWtwu60EuJMJpDhqhlV7zTHOYKPfquU1k8G/AVq3k3BwJwAA'),
    'evaluate_full_pipeline_tradeoff.py': ('4c09f9d1c407f77b3771764fbd7ca24801419ace0e7b3be2aeac78fc6f9556fb', 'H4sIAGH1p2oC/9U9YXfjNo7f/StY7YeVO7ISJ53Zrve872Y709fZm2nz2tnuu+fz05NlOlEjS1pJTuLm8t8PIEiJpCjbmWnv3c2HiSWRIAiAAAgC0h++ONvV1dkqzc94fsfKfXNT5Jcjz/M+8LjeVZw1N5xdvrn6J4uTZFfFyZ41VbzmxWbDNlWxZRXPoCVfs39+9/oDawrRIb0CMJzVu1XNm3A0+gj3+F2c7eIGWlbFfc3WPEtXvIIb2Z7VvIzxJ2vuC8Y3G5409Ww0moYCmjlEGTc37D6F/9KmZoBHmqRxxjZZWqox0iL/y+iCOtfxVoPQdi52Tb/LZciKHNDBft99+PGCpdv4mrMNjxskRcXLLE4AyoqarNO6SbMMbnwb183P6UeY/i3P/8LifD36ioYvq6IscGSDJMz/zx/e/3D3NcNngeoesJzvgLoZS/O0gUmlvwrExpKCm7SqG4BacU403NVIVl7tWcmrusiRNcltyLBxXJaTLL0VLREhFrdUzvYjmEpRNYqkFd/wiucJ1yDWaX6d8YkEfJcCywO24kmMTXBmIA3Qp8GBRuuC1ywv4KKuC2AHcBI65A3MAh6kOdvusiY1gIXsdZbRNGKg7ZonxRrwQd6MsjThORLtpw9X7xEmB05/uSkqjVuCU1+KmdVJUcmuHcuvXk8+XP396m0woj/s6mf4TxAiSUAcKoKCo+KsBVORGH+sSdzlSEVF5CT2jWCygBx1bSedFHkTwySbm7QmlDU0aWKhAPpHTV5xrFE7CKJQs195VbAEsK9iwPR6l8UVu+MZULTZB6wuYDQQoWJDqwJJt4Kn9wyGjUf3RZWtJ9dVscuRkCAKv8AyKoCVHTYhruzRSCzcKNrsUKyjCMQcpQGGhNmIdvVopO5V1yA0NVfXSX2nfqZ5XcII6vIXYK36Xe9r9bNJt5wGTApYK4kAr0b8BpBteBUAlTYxiMg6BXiiMS5UUBCq4RVc0oNmX4Jkqvuv832L6i/FCnqoq3y3LfcgOywvW1SKKlFQ/rXeKhj4m+4CBrD69k2atBjiOm1HUCos2harNONRmZagxXIeXa7LexxL3sd2vT73N/E2kqokQjUAq2MnRAQ6qvui52j08+sf373+/uNPbM78EYN/XhnD4iEYK77OEBRoLy8YepoXRoMNaJi7tImU+EVpXu6aWj1OS5TuiJRTtC+y4u5reDYefXf56kP08Yfo79OvAJfFq4C9DNhXAZsG7CJgl/ADbk3h3hRvwt0p3J7C/a/hz/lydPX2/c/vforeff/m3TdvcToL7LYcfXuFF5fnMFdgPXI7uYkyfg32Jaq3ZRateclBjvMEFIg/ZpO/su8Bwxmh63mgO0DuTT1xN2VlmtxmsJDADGVFvAYdASxZ8ypnV8KynX2/217tQ7EIEFK6EQv4Jq7jpql8KdEB8655A5KPV96YBhXN6XnYPYVJaDc3uyxTD9gfUFT5jKXXOainBQ4wgbmCuKyXAiLo9xhX/Zw9tiN4q6LIvBkIbYi/oqB7kuYNPID/tXsbmCXeFX+1+0kBNOQP8ET+0p4VK9QL8Ih+aE/qpoLb8L92b5enqL/0+0/if9TGOaiqgKGAc1TyckJh2vAtME2j20Y0FbSGdjC5KMKlHkVdG/wHwif4kJeBDnssGjXVvmsdRbS8osj3khtc6h614g8JLxv2Tjx9W1VF1XX6A8lJmJe/gi1FOVkXAqecgwwRmJCx1ywBb6HiYG2wP9gVMFQrULhxSnZCA4iy+oDzA+u6Q9sADUiMWVjeZmBrml2cgUuBQ4BSAWdIdS7Brknxr/d5clMVefor99f8DqR6RsoqpCtL/mE4uh+ihLH5HBi+W8deN1PqjDdDHfZYjoc6ee0cKWBFKU3jDJWrGLnZgQgt4CogQVvSQH2siQU1LA50LeZinBDgbaKEFL1PLSpeg6qFBu1g8sEQyIqDdsxlv8AFl03UuHKOYJTBTvMcVW+9227jau8LaapnoDfqZgFSmK/jqor3SzFLFMgFCjlNkv03rrXlTNcTsn9LZiER7EdAARASwuaD39xUaSK8ZXB6hG0H88y3ZaNEVE5GV/khYYjTBrQ01CXKY8W5+C5Os3gFNmZTwQKpfbjgGUypwx74tAwA9zV/EOpCTA7+EtqiG2ja/Lq5QdWzaCcD9yS0xS3fLxcCApKGTds2uOjhIS5i31i53m15sfYC895qVTzY9+Ss6959MFYlX0cuOOqZE558NgQXHeweXqBm+ggIwqT64MSupc60bfzgb9PcN8g4Dti5YpBlYIlKA0wi0WoZFXQMqrUbJXjn6YOmf53rdmTJMD38COaxqKQQA+PKAsAi2w3ha4pIeEuK/TQieyG52oqCgIJMOB2AYJkJAJhUZOT4HAHjIT9V58W57P8QgUINgBP1Lf6ygYCx3eLOCWRC+EOCD52wtkQIxEwCDZ1Rx3Nib43+BcAXf31wbog3MK/pWPoPsE8Dkb1c21hURdFE4JyAbgIdlRQdAicSroXcUQ8ooKEoG8DAlsbw264hzO0GPEN/MgURFXRDKsIu6CGt5+caHFwlEd6NYtz1oeoW8oNescTwkyYgVp8+gRD8ULBZpvKACQiVe3lhLb1xO4MLcC8vLYRtmm9j0LwPEaBUyW1M9Go9gHc3V2xPPX0XKWxEpuT7AjqvpAygKwhOs9icR8j3T6eeN0ivjkRuyijZEFOQSPy2xOlN002ZV4aN0xxbdEZ7ZCGZ1Fbbchzu8vpfO87BFJ6PgULKD9B8Ulz6LmhKJTwbYGs8Zgc49wzZb+F1rDTAWCrGeDbI8m5Z4D9jVt1d5/SkQnDNTj5yjKkBNTn89dcnjSdEedat1gNNUZpUU/ztbpqAMgehvONZOxGMl9SdYgbpC6R1nBv9n6RxrnZ5hLEiYmrOm/uiulXA8jz8UKx3GVd2Ge33bMikSmMtGT0zHoLriM46LEH8c4ppdrmEiKywYB3GGtZz+bcj0MOcUF7AUlt2t3FRtE/EytEeKvzbBp3kos+rrtCDFTMCMW/DoXWg7YuFFLVQlMAtrSYoB2YjUnhWM5QBs5mQEK1ZJwttO008ZMPOZ4ZNxjWPlKsI6xSZgbsO0U4QGQOiMxFmos7kRhxjP0E80Oo464HM8MhvkRirXXK9r0OMgnXbDXUnTPOaV41/HlhdlfNSbGGLswrBq8lgN74Vf9sg3921oECN29dyn+HOACPZuKmYU5tJARRN13zSFLDxA4qONJIo00tXymSFmortvI7WUGrEak03XT4DgOIjxczVlWgK8Np5aUulQziwbi480c8YV1sWGqq9u8Ndh0yfVIU6/mqOUVz7JiV1lUcDzcy5urpKjCx1hyEwiqZhECyTlBG/o3VaUXRYCr1zV+EW31ZRSgk+FrzrhFJGS0gQRSzmuTIoN+Jg5Ds2+9aM2BnbeAg9erxGNKpwB3yr/PEThmS8cQjeDYZ//LGxr6W2YmsLejKDXSMacvVLnszILfz4YAzA3FgKTL5/+4+PP75+jwgEYubRh9fv39IlnkmIW9++VTfFkYhnuh6wG/rXLsWDDkDx0Zrzk2f5CMTx2QHmYdyxC+WdOPtu3oQA0hEAPY8DJox6QW0QIexjko8go9abo67rRh2b+3eCMac/5qOk4hgEggnkdTb/Ns70RT0O0dj6umNo7PcFhrCe/p3Il+bypEwsKZAgEb4TS0zEfeQa60zMEQsivWpr02G6E0ZLEbtwPaeZa2GC4xJA7X7B7TDYj2vQKwDNBdsdcRC0vdnlt1EN220ZsdACht3AWqjNDCFKOmlWwdxEtzsVbSPSBVBbIrdGpG85yJ/11WbF4nI3vqArIKADVXpehzo9b0HawDRe2jNx7qi6aK1TCHqTuTwyGV1CcA+qXZ46BVCvGAY02TJmX8zFbQ3PZyjAjXfV0vSMQDAKnLFtWm/ReMzYo2PYJ6EZbS34aGNiqD5pkEBhVI0j0IsBT9oob8tfwIQebVbendKoxZvWkt7BWPNdFzmBk9sfj6lLFU2mXqlk8IuV6jbGtRpJBdwaAzEa2gJQmdcc/UwHewJt8WviUIPBRshp7hOYF1o7J6CxsaIjMPzg0aOHqc/IlKlVsd7TbsKEtRBDzhAHkPPZ0rIRWbECeEWV8rw51HM2tXqKRTS3tIXWxWqOAC9Ae/WMjb3k27nqfBmaqib0z5nnULeBSepq43Nm2BELnNcGNIzSrnLSobpt00RrblCp38Fad3UbeAOlst1ZdLTMXOBAcJhPbvg9kNYMpDqyDB3P7lLN0rTYL2YBM4/Ll+GWx7k5jXW6nYMOv+W8xJ8fqx0fRrsdy5jGbzmQgwW9W5Pe7IfJa1xPzHkcFy3r5vGB+7KmdTYH7/YKanstLDv+FxkbbP85mxpj/Yl9FB7Y7/DMMt2mWVylzZ48V9DL20iIX2DbO5lMcJ0LA3EqGJPdNtcCkxcu1ksrG8Yl7vZMcH4v3kkrB6gSZ9ewlHGfEOH21VeYT8wRAyGBk+m4B4mEdfBpUu58x22Kd/YffAn+0Pl5eG48cEVcNY/hc6fca+lcSgME6XX+P0Oh1ln6nenTX6Pqzv8DGrl8RUUvWwQIC7c1MnsakmJ3Q19Ps7yBaVddlrBnykYH0zN4Fpf1kHeqZWYcNNoJbEic1BkbVHdbZOzcp47Zs7e5IRhj9tc5e9k5sEZ66Jz1jiqlvLoF1e9b9cnFEkhwASLSezadTaZLcJB7Dy5my76MTZhv2fEOtPlAwTXvItCgB1WcBV+Y98e0OMSj6ZjAjUxx97+9+unLL3tHU3hAMEhKIJ7Ig8ENxRrPm+b2GZeMunR0fbSyOeKI1PB2S0ly+sG3MkjWJL3hHppKtzuhLnMOcjfQQcy1y6+8PN+UeJ6ok6Dr8dT9lGvHPLLYxrB9srK+RBpuhZFumZIbvq6ud1tw8a/EE1+PZoKHKlIPsriu570Obyjbtv6OZ+W3qrEWwqahwni9jmLZxfcmE3F7PcFkVy8Q2Y1zih2ryKTwEg+CEMnrEwAwEUc6nwilbnaYXT5JbnhyK9I5PhUSnp1M8OzkswB8Ph6Ucju55+n1DablfiJdtmU2EXvISRuL/VRYN5evthOhOSbtDueT8YKWGDRtJ4ahQpXzrRJRBvrS0e4ndMQt6kQ4wxMMPTghKL1zRM6OQHn11XGuHAFx8fLVQRhkqj+V/mCQWx44QMnjtepa7KMIhPiDQPAAR+pmGi6i1FndMmIz6rEWqfCB+USSMeqWidWAjrmHntLaiOTasB6imEb2RtwBHdc4O2MebNzOKOp1hvfDcu/ph3fbtMbCFwy84XEDnjuMRUhMFA5hRMykgjyPolNZday0VO6GhDbT9kyCuFymisrB2oMdAXPGPMwhCZgX4rx8CUQd7sq0azqp0c+WsEd7stRe0JmSZyVjA8o+saY9HbRObx5xkN6BmZmubcymn+DSzq+DJY6uBgd+8iz3mY4Yiu02xRiZnvNd71ZlVYBGqQE9ITnSm+3wWHjXKYq5V/E7Ml148d3b12+8ZcCS+/XclA5YEPyh6QIdY5DbKi0751Ei8oUV28dCnuibHz58ePfxOUHxtw9YNaAqsCTsxyHITwFwfpev2SO17J33yTQCc0qhSG68A8Y9P6tgEJJkC/n/rQNOlz4loiM2Wgo6nmiovGXwakQSiQd7FE8dwsDq9TfeG3nK9Eiwnjw8K9rVN5p+E/V4zvy2rEAXLeoa0ASEvxGBSqLEom4WWl4ugKLSoRAPzH1bmY1N7Sf6RLd8bxVuWInLdvKzK/F5KOl5KOH5ULKzljDqzMRyQHH2MB7aPe1rO5Hao4ME/c5daoLvZ1l73UToODpe1VhXOGe1KFH0XaSfYK2ITLwbtyuAeh5chRtPeMtrKvJL61bjd4WkKc/WoIUfCdqTJwVenFRJ26DlK635g5FGIO6AcY7vI1HnCJPFdcdhJyqqXdtcT6SMtssD9H1NvM/EItShjMOqLrMUDHoEK2OKKZ9CM4P69PVMdanfFbqHqxW+L0iLS3qI+tGa+mINIZYZxveyDLZzKTuCC1WuvDv2V3beDScCAPIkyGxHp0AKQf38p6hTqg+cy512XcZJewDV9sCahIAG0HaSoEWUJ0bFRBkpWOCX6rlATaNGGS/JsstL5FKLwNKxm9UAKniyMAmIRvGOOlrtiWHQSBY3+trRWGVyX2WH9jnbiZKontWHU/kdVGmYpdfpSuRgPB4TSqH+5Rx0sXPiv3gGsl1sYD6XxSJPhiiauD4jdwYrb1uyS+FU1bZ5YdUoCwkWyNdUs6sqlE23wjMrsQl8TYXfeP8G3Exew+ayyDdYxAxGjgYQFbhgpBNRJ2tDXXFVh4zQ6wagxdeAZ91Q8XMJTJIFtpjC01ZHh73ja+kuB+oHBu5BHQvNY9k8zO+iRv5Rl1tabE2GZAqpE6wj83TIIwjcTnx/POmxkZF2jmq0GHZBRuoIsj3IxjJdv3VbzB3DWM+Igra9tLhBh1Sfo5kcJHcaUS4LMDrPwbUhGS+0OtqAzZauxBtX2YA+UJe0MZDQTtLTFp6ZeandLzuPYbk0lMddXKWxOLrqSrJ97EP7IPkYdYkqUtYWPKxjkFMj6UgDo1XXnTKoaHjKqFj7rIyLTKMwK620RArUHhUWoq+jdZVumoM5IqS32rcjRAO07VEUQPSoR/SB5Z+p6vG62FWJKuSDLuf9JmveqIp5fD7q6XOsXfeVhkRi18nc00y5vvFr3ZH5sCUaGeUJtcirdVYcygpDw38R8i87Gq6AAbAtn6sDvYcBSTb+t7kNBdmX5tpRtHIojxa1da5nf7K4/ekBUBIl9xSijWmeOnctaIkbdBgtZlrB2NJxFiTzkymDHathrdpBg9JdkYPnmdpIz1h+NizVUYNp46eOieRlzYEJgt5URmxmDcgq4izertbxrKu4UOUK5qRdJ2QKIzWsuv7McU0KuQZus6yP5lopNLv8nWM1BsdtqEVskwqHsDXI0j97s2Y10rxRchHaco5W/DuQxoLgD6hTmsjuaM5QeS9WpAgWz6KXqHR4wZq1o4spM9aTNYDuOAR9kiucKTlCpKK55MdBZ9VTMab1rIqKnySTxr2efPbCZr26Hqe26D+2uXK87E3M05lM0kmvGeQ49CaSmb4igmNd1OtJZpbcWzXZA28tmdlc6Q7WurNyckUWQ/guF17r43pL9qJ30mtquxc9PfTCXoKuPKxhJBQF+niYI/dhDVFl2T4iONZyfhYgCyWX0LfwCthHVXeUqQ1apukVupv1cLqd73a4Q1tEzQ8DsyV/YRlbe9Hprx4iveN7fecsqmLpt9axHkg+aTcbgfO5U+sZ2s/9iIKNCHxIMz1LjR1SPf00mLGLyvLVY5H+toAux6Kf8KBXYr6cqnCM3MnIs/2WVw9Uf45ZN1aqzIEiUwM56d489hARBcbtMGBqTlOAekGxJl3PBKAVEFtS+UxAXanuEDtO6C/rzXRBV7nU+OhYd1mOq3eX4Yo9RdaH+39GjW5fg2t8V8ZXXh63vaf5hIZUHVobmkFcuF+WtRTZ9BqyRv9W5br7Ljz8JYthUd/2NFm/flwE42FPBhsinrdWyQxKLZ+HhGE9TkFBGZDfYnDL4jg4PXIVzBx82cKzX7hg+ZjOt1UMvLFiKOHPrsr5DGTpuOV3xVaWAZoRAnmq00WAs+Le2AuBJZcd5SmjrD0zhnMexvwjx9dm0asoRdhCwpF1g19UTyLe8ai21+1pDAXk5Q4dX4XoDnap6M/jkxEPl/GkQFvYIvzfLfP+S8yEOqIiv4BK/joN1K8AdGuPvsrURHnwoeB7/6mr1rELdPbvD2ZJHPMauvjokN8xdq51SWVY3KJz9xrKiF6KKX1LjZSmwBT3PU/SafnlOGByFF/7bZSsYCMVqnEoNEo2UsvqkGVCMSKOR9qr6uhlpyQIbhGyYrQdkTRgS5XWK9/+1QMApDHai3p6DA5TB0rfHI9VyqsEQyfvhg8uV6+Kr+Kr5JwK2jGHGrMFHD53Si9F7Q2j8Pa6oKawmn1DY7Q4AiPCM9VT4LAzyv5FX3x8aP4D+0VRjHjafuVzxeNY7PlkUdED46odkG9svXRR06KtOAJB4brTtSJYbkeEhAsinjhjQNITaqtz1fmFSE8PXLvt0zqMe2n4dlRfTVZcGHbqoLyarDgQrH8xHxKmofD9C3xRaOMfEVK9TiDNLariC3bDNczcYWIenUJ0mtY7rvmsQI6ZhW3Kj7PfobiRs8NyYaR6LwfQMVWVWoziVULudXoYzoF54dtd/eG5ud+fq3sd5mYXoRkzHB/GTJORGRvGw6lzjwn/wU7CYjxXQ1uBRbf9tBp0mV69iKTKsdRP+Y4lteRFPhG1BkZeC3it2nvovTaxFd8qWrt8je79GI4TSA0JgqArTytoKnU+cK+/UunZ7MAbS8cHzItuWVx+hTIzpitjBVpxO9zu4ogQvu3Fje0oq1jQaCfk7A9EWtX0l/JFQuLB4a5afNTs3el6HcCBWKbZnVaU3nVoR2r2sw2ywePnsNDNukP23mAh0V4YNjXE4HszFApWDYttLVVCheAB7B2a2ExtNPWiaGbpZyRWnHtYjXT5MvyT/q6+Xkdnr5evwj/riYZ31mB3vR6v/hR+pfWg8pttJLTERU9LSjjOIh0DavinkaWhiOCruKZXvVe8rIr1LhHipyXHrGpfI6BFnzEeZU+18kDxHQK7y3PbK6qc1tqiEPU6Dy+NV1eJCmaD+XVyw7cxFgLWIFJA2Kn+gqqkEC+osrTdOoYNK8cNmQfbzbRueNVmGPIaXwlhfAZBpSDiFzQo5mC/nLflGMws0sSXErMimeswE06ZyoYYnwij4jJPLGp9IIek1jG+R722XZFWeYhMNCc+lgsydgNwYEFPnoeG7qWCCkyy3Rowi9rvifBKBGaHHNvey5NL1ECoSsACc6fv4f1gsBJFT3wdpNPS6uMoxjcPug+NeK7Nukz5O+GzJvgRGP6AX+VIG0LBAdD8wAm+NbgdwvGdk9r5nRMHWJlJCDhpaYQHMggFwjf4TYpCZQA656++ooKEEuai9/UUPW3Q4U11wt7O3M0889M6FY+zF397++Y9MrJKblJ08sCsnOlpfe3SrRuR7gisvYLt2pn4YI6LSHRgLT+oY3zYJmD0uZJJU+2aG/l9G8qnQUoJJ4EOWYFJhydsGHH3XOmTO/mE3h9YAh841SN2398RH2ARaFDdGegspqrvHVNTn+sRr7IjsC1baRqHvuhyeEIDITPnzN4b38HAr8HITxEAr9Bo4fcwaLHhqqFPHCFaWSIcDCHmjumRg6fCFexdo/bKWACrPkOjlodcC+qTKAzEFmHnyf6IqIrv1kTquzUwQeuVMo5GB7SR8xM+THxAaIANSApkU8j+ASv+7Yc3f3OqJMx5ZhdiOcaMvpzDhj+5g6/fAL8Zd+yHZ09e9oEJ6a450n8lc/P/I77GN3Wv0/g6L0CvJvhhqGJ3fYOvymGtUharrv8eLQFarlz5qlvxzsEkRuAq01myO8byBVikxQT+KK6eNMUn3acTzkjn4AIlI/xiknL4tSLankNBRgr9CaGnvvn56kd2cX7xFfuImYn4KRpUXUz6HTUrYTfJSJH1PghgOLOmt2p7rKZXanqmpv/p9kFf6cB1WvS8yYZ8q16qTV2LJeHwPm05KrDKnPT8o/OwUBQ2qEMKmgP4jI4geG8a4CQGB7ePW/pW3DoCWd7VkeC0cqPI/RwQiaq4gy2JC2t1HojFbvQZG/gRuJq0YoOvR714+QpjWEYep7hLBYzuow0LzmG13Eus/+Rh+6BOGllVA2hZUOpWT9TbIrYTkOzVvVnQzKT6T561CebgjF2p9J88rgvYwdHBxqUbzHdc7Rvujty07lNkiRC++MEhWSFGz3wsLB3I3fGEV3IZqWiGYqwJ2j+eIDQ45uiEQJ1kdU5vpCz34sxacUxNzeTjwZnZ6sKaJH6UENQSeLco0jvHW1XaFhj455mvSrLlXVHYRBMPO2BWyGvAKsnoFg4tA0LGZ7TaXaLKjeibJrUGOZrOKurORDCQpodpbKXZ6cqB1MiDoUKJmDwL0mMYEoD95EAMkYrJdcsYaHYwaK3eYBBRp2i7oVWGXpuiuU1unfXBYKkdFbMF6VnbbQeSPR7U4qMMp2CpWCF36X022Lgf5Mfn8MJJfNy2Q8PkBt/gGd195iT7S9I97V6zM/pi0hFSBGzKJ392vfFoOvhKrd+Bau2hhiOipR/ADJzxBS6ftQvnnBx1ESfag6OM+uQ9AJlN+wvgqX3dRyjfrAlKE3Xn9hZriemiprMYxh9S0OnFrVaEr/e8r1J8tTZ/aHzteJLiiFRskjfzizG+0OK/cuADcL/AiNTc2zWbydeSH0l9J+pr6OVGdagf9agaC9x36QOnzQ1ILWxzHnwvBAASlKjezmWp0aKn5/W68/Zw1K5PN0raNc677lL+QeAOlrvi4P0ody+KbcWnA624W3zjVpErLEpQeB6+jCnn97g/mHsuGuMHRusGdkjb7shKMA5PbwBY+CZNmn+KGz61CzQ6zrufKA6gbOtYTH3u0acttUVFUEkqbni8NrLE9Id4purrbNZfBTE6duL9ODr1dHk2cDKuFBtFeo9psd/CWjocoSM1DocxH3YB/lfRd2cLHcH9ROPxe+JNKoo+a+prisXlDutyKnuoJTg+6Okq/ReMHF6ofeKN735LNywSaT1RJHKRogjfBBdFMv+KXgs3+h/b+26ttX0AAA=='),
    'evaluate_mobile_pipeline_3dpw.py': ('9305461efdbe5c1fa0893b25536497643d636f5590695f3f8f56dc0820379ace', 'H4sIAGH1p2oC/7VbbXPbOJL+rl+B5X04ckPRb4k3pxlNnXfj7OQqTlwZ10xd6VQsmoQkjimSQ5CylJz/+3XjhQRIUE4ye6mUbYJAo9Ho1wfgv/3lpGHVyX2an9B8R8pDvSnyi4njONe7KGuimpJ6Q0lUltMsfaBkW9ynGSW//Xx1Q1hzz2hNipxUdJ2ymlY0IRdvbn8j6TZaUxZMJneblJG42JYZ3dK8ZoRKquHjJtqGKxrVTUVDpFSndVOnRR6Uh4DcbaJa9S0qkrIig0Fsgry8jVj9a3oHk5ZZFHO65DGtN6RYrdI4jTLyQA9lkeJ0UZ6QNE9raE0/R0j+B1gP8FTkdNIwCj3If398/5GUBaNkVRVAi8KYMmsYXziLtpTQfZkB5To7kJw2dQVTmERJJFhLP/6CogrIuxrZKypgoSpq3mfKSuCWJGm0zgtYbSy4SwpgIi9qEmdRusU5J2VU0urfGfnl5vY9ubn9r9vrk9tfrwnI4bGosmS6roomT0DWW1pXQCfA7ZpMgPstCcNVw0UawiYgAzBJLjlgk4lqq9ZlVDGqnmO2U3/+zopc/c0OTP1Zp1sqZiijepOl94r8LTyKF/WhTPO1ar/KD+10vxf3MEI95c22PIDESF621IsqllRu371XJN6hFknafyRb1Yx/i9Ymw704cFnKl7iZ7bzPKxuywV9i18lk8u7m6p/XH67vwpvrqw9kLhgLapqzonIXp8HL1698Ar9eXfJfp5dLL6go28COuRc+OYP/Xkfkl7s3Fhrn5/+Bg8/PX4pfr2w0JpOEroQdhXURytH8eSYEE/CfHpn+JGe4431mEwL/oqqKDjB3XgYR4w9irE8S2CY6h/ZVVkT1xbkXgAxzhvrvngNDODs5IeevXgWnnFRFQWo5ccUkKPeQ76DLycL8xBAaDtYFIFfC/mgikH1cFaXLyQ6X4vP2GMyZVuF+RjiDRuPBaGRpQnsNn6EB7B4Wfg47NOHC0WYQsmFxBB5szscDs5yAi2M9fb2cPyGbVVFtBdP4j3f1+WSe37aKWe5U9+Dq7dt3H6679934lgXfaDoNTs0GJQgQsOT0vN9lMMZCV4luhIy2BNDBCB31XKzlk3gEkw7+/u49LObqU9d3lWZZXGRFNXdPudKcSkJKc7dFQrMwobs0piG4mwbdtCueZ1JhxRPfJNy0/yWsluor9+CUpCsiegWotmQ+J07cJJFDaAb+2onLBhzf5D8FvTRfQQjKYUKc3fUkIxiwwgJCVbUTblDsBWo875jNuNeQOlQ3CXA667xCIOPNzzefzn8Rb/1Og0N0h2xGMoh/C/SFS7+jfh/V8SZs1dKYwf7SJiCpx3UDe7PQTd0nx56SNK4XIFEfXfFyKQS7D6viUbELTiBPuBUvwRwWS95jG7GHZzuhGasO+qywhx8gsuo9i3wFagebovpze9N6cEkxCh0TBq2g0xPeDmYEsorAjac59/mdCYGVrSkqXkZzV9sGz+/LXVPvhLJ47vBgryuD0/XIaLSj87cRqJY2LGXRPRgFhmiIhwGrE1pVAbTW9cFVWj9r+xdVuk5zoCGXq7kfbdGCoZrGIL1QZEvP9keB4DJRHtqiF0JIMymsF30RLGeGP+CZkpikKEF8SMPDMMiKpoqp2VlfEHpM3iWArdrRqnadT//8u+ONDmABpEKQTLmqAQZC2PCGI3qSUOMG/XTqGDJhda57+RIUAX7A5o86ruGU3qRt4nKDbGrOk5wAcq9VGEOGBW7T9XTvCOkGamjnOIISMl4wM5PT3mpMd5xu1+zzHHk2vTSYyRxU//TM79FCDzA/5k09c8QmylZzq880+8EO3sNC+ureLXjM9ZoaIqwfXwTskMcbyKBxVzoyhnm/sMoYQ5PYg4mh7GqrfSl8VPzPadnqE1MvmNfX8aTe+GRD0/UGk4FWZ1BjzO1YSRLBfbGHRBwqA3RgmGujb9HfeSiC06F9dB5Oei/9JYapY0NCzFApapU+FVrYCrx/HcUb1xOZGvyGaAc/RfI11Oh7yuowhbpgD9QgoLiY+1XrbbR3B/NZbNBYhphxMGzRzbHUDEgKUqPwI0riFUpRLquryKSEh0IRoUnZPjD/mVYFcy/+ZslZh+y3YUsjANMw9+zrxvOIpsYig8ck1Hbsmqzd6zRvqCknJYgwieoIJD10cn2JBdhTl/y3KIZnnRz1zWBkMfPJ7HxJ/qoXDAPOFtyuIIME9wU5pLIv+bwcyvkrWAkNvRswdb40nRZU3IneTxv9E+rchdEdjCncH/aHY3IWBoe9vlfG+O+44J4Xnk+eka2VZl/eviXMjT+ByaKb4DINWLOFyIxC/NvQNLdpnm6brSZ4tuDDlgG8cqN9yuanNpvcjw4DrzQ2TBU+fle7wP65iokXiq4nahkoYAvMnCy7wgueOfZ3xTaCqPBBMTZVK4OV/5WcBVD/ngU9jkY8uCrOlK90la4tTpfAYvt0vlRsjtE4WGicGTQuxmloK7RqiCCMrzWOYN0at74+kf7qbGnxk0LZQUqv7Cp51i9Je8mW8IxFlUA8rnngg3UBxSH7nbOa6sa1GKqHxfP09B7Eh5IyGrMCighWt9JHIRmJQ2/1WRELrG9+3ElaxSIW2WrNSTv7tDV82eI/P/5gjG89xVECsvYf7WPxMMe9iykcM3gPSAER8NQgP4q77rqaBnTQ10vI4JWQPQ8kjPO7o3s6ORr+XVuI+BEjhI2ySczIBfoQnA5kdQnqUCl9LnKvT/p4DiEKX4FTshY6FIkQ5qOcMxDT2SmCh2KDJC6ny4g7WYxkkL0jmYVICzGn5n/5fImYTlMIaLTCPRGkeR7H3wlQmlfzOle9Sn1QqBulucEHsD2EPrTEvWPXGDYob4dEuhyBt7VygxHxAzgMXJkQwbKTAa+i5ZRy0W0nffHegDj/DUFH1l8+9M3De9DcByg553dVoyWDX1dbFk1dNuiJ5NpcPsW/vBTrb+Gz1VhvIJcPCgyBCcGzmR5NepBPaIGhNQsYc9sKcZ6YK9OgZxghNlf4HU+rgcf7tj4CnMt9UWSuNkqtsGv5YsjWWVXRljJnNoSczH6i+OfAEof1XHcgjp+wNvJkxjU2XHNbjg5GwqhtVOG+9un2SfHCm59ogZdRew609IK8N2QFnnGX1lrnnsp0/Z9MuFcei4X8GGEU5R2eU+AC6rQ+tDZLD/wQxOLafAWGSIJCVeTRFozvZLSN6irdo89Wr8NLcORyqu6w5Uwctvjk0tA7NSig+zICBy36oMO99AJe1a2bomFue0gjCcNsRf0ti5fzPcd4T7W/UUL95V56zwDm7aHZDvQsyiW4lYP6ow+uxL4/8qSDzfjxn2jKonuKqKeJO0vMm2ag1jSRGCfYxVLH0FFwOqlvAOFzWj8W1YOSd54HN0XSqEOQ70ThubGHWbpN62/B5nsrF2vtwfASje0wRNBcxKRd6OJKqcqcIcqykFZVUdmQdh1Ah3SD0T8a3MiZddKubw16lYXCmWEI1xo7z9V7YQHoNVKjaQHfuxFcv43ArWK0Sr4DcypCXuWjSIRWLRxodpYLBT21bptvVMon6HQF7Ee4fjVYdbNQiHZRmiHAj1UU1LJmyg++vp0CA2MPncX3ag6UUkff7GtEck25MBSYMbzPjtbZ714a9LohPw7wyRaDatFYjFmGsJTmhFpQc3uQtbJRv90dvxP9Yqax8IKcLS3LhowTo6/fxVn9DCaE+MwxqdGzOgNM5mbTO/fUXYKxVn84vjN5K5HR98riLbsqLoOEOr4ms7Godzjg6in9qzN7Eu+TPRTmHp7hbecdoNx35a9fdyxAxramdTuvlv+YyPi4kZgKvDgj/X1VdZOtujsO+HWMtxRFQL345mOYNlXuVlI1ubjgERdVjzsZHebyt7mf+/l+cTZbBk0ONR2lkFJxLMl+rILqO8cfXz9E6fq8TZ+/eigqVPhQzgeKdXwEbuHckorZ+qI9z22pS69zDIKFmm5Hs7muumeaO4Kkop96/D8eKBlx5atLGBFIDb3hzoC3h9E6SnNWhwhJ4mqznh4JrVM2YhY9vrQ9y4q7CK6KffGk1YfK+25plLdQlByDbfpxqR7orVDLl4FpOjKJg1QeM7ghyOMogtCjde7DXm350236sJNR+wz8+0J/b4GbHG0/EroGEoZwzAFPFmkbuQ0ohoUDuYxlb4yW+tjH6Zz3xvbPNy2j7ZVYn5ClOretoFeiLYUDBdtCxEJLGbXLEimj5BOYBhjKNb50V86HgkhBqXQ/4YnZF9SSJ3mo39pMF8u6CcyzEJm5huDlz19dmgkGbwpXENnbBFfm05kW0Cdf5yI090C3JXiuOIo3yj0MgIMvlpKYz4Y1teAlGNqF1k8uqOsuG/oDaMVgj8Dvxg8KJNCttV+bi4sUUlfDNI+zJknzddjeKqWVMzMU+hjM0FfiUUwBgT5Lf3LCDxCM6ciZFU9o61KuBAYy0b4SEMVB+boemXXZhBiLNZDhy4gvUCbTsjwOXejwRSe5MfCiHYHcq0VYez0NEI++I+5de4M44vKSsDvk5jdtK7AJdes2uJI3OG75G1e/9LCNaoSP4yxibD4Y8IauIrzv8DPNyreq86Rzg2KqIEqS7paIM53y5mR6kZSPjk94wsmLbTCYP5q0oomGWo6QqDcVpVMgMMVU4XupyBR7CiYbP/CM5nsp4ZZN8Yr1nyLw5/k4FFmxez2V3uHPUDm//LNUlEq2BBC6AIfKdUadtI6MlZHx2weiTU55yTTFkslKQQG7z2jFM1QuXx6lInK075UdmHQrPwspichUa34BTJDgv5AIU8i3ms64b4AdRN8kRAv0zTeq6uxUsddBlDZjb4X+hVJzLC/PL0decrpoQeD+nSy9P+EBj51ge1AeHF/zK9uUMbzbPyeIKsk7g/p9xHblMhHB9gByaR71vaUK7pLOTMMjuCh5qHCdGzlNSw2HsxlxoP6E3XCC30ECriTiSaHL0m+LeMqchKG4+x+GrsOa+7IqwB6YA7k6SjAUO9ttzsJZp7jRTkV3wk3iw8/XV28gRSXxYzI3ZQUqQfc1VyeJroI80lLmH7BCnZu/6GkQfjET/uPjzc27u6O5mVlCOtf7kgNk4osbSfiLleyTD1vS5Il8Lfo+Ob1MWeRXbVInHl15qxoWoCVYWAupdB+CWnfl2utht0a+x49wFZzE60ouQh5AQtB/3oaAQJHtqCpvBAgBdMSnIkAkSty+5Xg6ctudJkLWBXkT106diwhSbXmc9qzFSaloGi8RgyFNC9Jgqki3NN9uwMPJtuj4pH3p56F0b9yHlGe3VfQYtiipcYZrgKVG4e1qm3XCgVWdihdUrMxScIShg5kfgk+4+0mqoJelXmUodo/qMZYYYu/EJ2EiNxZjoQ2/lgIWhBIRmIhfmT04rSGJrVIRrYdtKtiYX30RzEyGl3U4HiCBVJOcOJ5WQz3jtmqqMPC8DLI0519tqQPtdgTHd8UE2ikm7K4KUj0uF2oknn24ahZPnEerR9zMlgFZ2Mn6fXCuAjS/PMkjsPau/ijsj7PwOkeVMTiV9kmLI8JIzjNsx7eFFi2PF93PL4fdu2AzvBhfVrj0lfquENVdVJv4AeEXlK6Sl/ekq0wQBDDFKmvYpne4Lm7w+UpG/GwX1mw/wGohwUG5NwiPnUPqAcSSP38MHn8ek7YCkcLPHYOmDb91tJOtWhwHrpV2LVAqy/YWcidhvFEDAQ+RJl3KXm9XjenwC8IgabYlG0LFX6yXk55Fqex1rGB20X8xcllyiC+p8bbKdrlwEOv4naOuvP/SfrOLk0aAyrG+tjDzNGxCz57X8/4Vqx6e3JqAvpXCS7wWbPPLM3JLTZtefscytWUJN7ID+7bP0zqDf8k84tNZdHFdHwZhYxuFO1oxIAu7p52vwcuipAMkwcFbxIyiWjn9L5RrvDwnPtCA5LL9cphnWTJsCRfg9KEb6QRMsKd1Xb7l2og4vwsRNeCDEEYds1XH/LBYsA6MrdKK1aJKEx8sn7854Re1XhAE5GFRRAgXGtQ3yvz7Yf5h84vuooUo38054wgTCJyL04rydZNFFdlRyOVgzA8Q9cmb218/9seJ749DyAlQ+M6V+kgc54DiPIMl7Gh3R6P79PkHnkfcXk3F183HvnHGHQkcKxYl0o12ei6W28MdJrBtnIsqShom0U2pDySK4wY2AhZWbg4sjaNsmt5u8LuPf0BiR27eE/zaPI8PeBMMlJM1qDeMgmJAe3bQ+dGs2YFqY0fzSKDpPV3UcnKFOIkn39avzRY1ANKKqNoyzAHaOEh4n6M6cFAjifMxd+X083NE9HtN/W3tcv2vWrdeG9h3RHkpPN2Qf2pvZcoC+WHDQpnuDKJE5/SmrZ8VNJ5aUCCQt+GAIVhesH3AzFk8MOGxCd2D/wmLBx2KwHvrG54fajFTOD+/jQkeFr//I/2jPtljlUKSg/WoqyjBNHlcIJQ8d5p6NX0t8+mY7fjJvgAhWaADmLImRtMzqKc1JBkNOMa96wRAQJLi3w8qeuITQgdRxZw+QrZM545jYYJ/X1iDI9t2+SBnHyMJEAveQOb6G29wRT/I+FKaJZgOsDm/3o95BxYmXo+CkMOGRolxUKy/xKFul7f0cpaxfMUCS4/qiKEXgyGgQeNqNT7me/XSoCXUidte5Wob7NlO4DS1kCPUTnujuLg9f/FaWBxquTDEfQxDfqAThgiSh6E81BGI+eT/APzdfQOIRAAA'),
    'evaluate_wham_feature_substitution.py': ('6063f5dd58480b0c0c7bd144fb56f10e4ee206a4e0e740ad8da8dac30a08dc11', 'H4sIAGH1p2oC/809XXPbRpLv/BUI9uFAB4QpynIcbri1XsfeZCvOuRzv5kHHQkHkUEIMAjAASqJ1vt9+/TEz6AFASkr2qi61ZYEzPT09PdM9/QXsn756uqurpxdp/lTl1165b66K/HTk+/7Lp3+bNKpuvGKzSVdpknk/vH0/85J87a3TukmzTK29N0nd/Cv94G1U0uwqVXtpXqdr5f36w8u30Wj04Uq1w8ukqmHI6ffvfvU2aaa8eleWWQqD1qpRqyYt8jrkSQw6/TNXuyrJRmmeNoAo/ZwgbEikXFbFLl9PmmrXXHm/vH33k1cVDfXXkfc+ufEqdQnUqsrMnG6TS5gyqdSIFgEAn3YpdjeFtyq25a5R/WUVudfAWtRtsmq8Otkqb1PBvzWtMcVlNyrHWZMs23vqOsl2CTCPBlVqtasq6CauwByV8m5S4POuYYqTulYNkPtjM6pUWVRNbRcxqctkpYDhyWVeAL0rYEleNIS3TEpV/UftvX33j3evn77712tvq5IayN3CXEAZ7OFotKmKrRfHmx2uI45h+TgBsC43bBqNTFt1SVtkfq/qa/N4ldRXWXphfvIfaIh2wELT+ltd5Oa53l2UVbFSdW1b9vaxSbeKCVsVcIp456PkYmWoewVcTC4yDVQmDU5uOt/BT+5o9mWaX5r2l/neLuW34kKQm++25R647OWlIGFrn4tqdeX8iPI82uzyFW8ojnzDM7778Scz3Y94jjQdOMa057lu/LTemjZ8Ho1evn/1w48fXr/68M/3r72F52/gkF2nTVwns2cx7DOe7fhqW83i65k/wrMSv/rPt29//IDAs4uzZ5tvvvn2m9NvT1bfPnvxzfMXz15cfDs9U+sX35xdnJw9Wz1LZt+ewZaP1moDNMW07ACPopojd8be5C/AgihfJ1WV7OcjD/5LN15ag9A2Sb5SDBxqJnxQeV1UY4bD/yoFhyj3CCgCmU1WV8E4WpU7+JcnG48EHEyV1DQV4x1r0uqrZHb2PEYVEODezmlLibq6qXi6dXqJqmdhTl7Eg/QEKD10LKKiVHngVxf+GHcJhqtk2xK8KSpvdbXLP4J8eikogSBLthfrZK4hI/hnHbzwnngn09kz/Wccehe+L5bd0hPtyjWIdUA4nbXq/it1y0+BWSyI6Ao1Q6aZW8/FFoTep7m3yYqkodXT01yipZYABvTQAGvh+CvsI6Dnz0KQpnK/eJNktYI1fBpbfu+226RKPw9RQPOu01VzDhwJeT7vv1GdLZmQTZbgNjxoUmAnbBP0T04c5txZVvqgOstM1f4cpwgQeVQDZeOwBQEllvuaLQyBLcFYwpRnUwARTEG40DubOkAgDQNA3545syW3ncmSWzvXF83B5Dat4yS/zFQMcrVNmiq9DdrGuSswyFLZwIwkyBpYyV1ZCprlMroG7VdUcV5UW4EwhC3ZLiYnofdRqRKfP1QoP4jnKsk2sUWmH5540+iMuust6E7bAUq1Dsbed96Jmjzn/lUCN6+hQm3LZh9n6UcV8IBxC3T+P4RraYFBTQRidtM/9p56bovAYVEAfd7EwHFrVH/awTUcIIJnL6IpDfuE92aVg+K1865gawL9WNSSBDjmLdOACTTn2HDPnMEE+dHiPY+iKPTmJ0wmmgOwFdV+AOZkzjDNTREjs2fRFEhtoewCIhAxe+jhWlcVQFvM0S4HQKU+k2AAmYM9Mx69qoq6PSWfFfzk/SG0DHMbeqA6PnfmABNubYkgNLyKKSwEd2DyeaBnhj37bgccvCl2fB7ooBGT227PTA+Z7Ad6aH4eAsYhCGOzb4/gXgWnsGWoWBbtbkbUAO3qOl25HdTiKBiL9GveK5745yJX/O8SmB5YmecNmgjmdfdxaLyZw6L5mo5WC4pAtGyCMMqXlQWqDWPSxc/XAbc+SG/oJfIIfXaBpfNltMpg1qDVuk8YJqJf5/PJbBl6zw0dYnahw0zrgyjZpFUN+rNWqwIM74VFqYk6hencplMtPzQQBryJUM+hAQ+6mJFJSbWIJZxlt+6deDwUpZ1azKZ1dCX08xSj9gonlap3h+T6Kq3WrZrBvQucRXY1Ce2EVoZg/XwMuuCE0SigmeH9pSrWqk5X8VpdVkrVMRqIvAVgIfMSS/BA0lV/LwAn2OWqcVtHBw8LXGHpNSr4FqH3V40jaqokr8uiVsQvq3IK0Ow4JAjM+AgdDrR9A1jJCSyFljRDVhgZgZ04iaaou0EtghUIllXZ7tYE+kIEEOx2+Fcl6xmwQ+v1BBU702HNFqQzVlUF12NymaCJGrdaoMs0ONbDfIsPXdOHWCj4tjggNc60wugBXfcMRU6fLpofr+Ih66FHXQ/P6dhh2z2HqKXJrNzh47pKNw2f1gFW8fHtdRzTAweZY+Y4yJee/nDHW1oOIngYQxzJRF7ACYWbVXv2GFb4pdmhTg/A2XtbrHdgOWjfA5gWxxhsiGMgJ9sQH1DFtz5BvQO7EnSvhRsLRZVtogvQDhcFSRW6mqBcFFgN8RZIzgLHs3DcQD/E8wdiCoKwZqs6RPc1JuJVvZjase2Eqyvw5lWGVoMzN/pksYlguORZbxP8HxgGHPhFgSGSY6NLHnS9KvLr2Tow04BYz16gtq3gV4zW+wI26CJNauN7dBH8vSp25c9o4p48p9EDIK9/+mfQb365TkrURy+vL98VRQZUBCwZPcg3oLga8AX7PT8le1Xx7DN09dDPOx0AA5YnlYQJtTs4wHJiooksxRe7zQZOg69lmhyYUFpwASF64PC6WdvRsIt2sD2b4NXeJNWajmbI8awHyS3Jrglodc+KRmrPS0B4z+dwsdP/Tmfzyels2a7BXtFrg0seqsDgGXejB2LcEx7XrhtsKtmCjNRKDPyzNUCwwPKtdaVWH8sC3Mi4DSIYe9Hwg39pVb8Dz/O8L/+h8H9f5vuldnxb/K3PBkQEnXlDsMzKOCtWpMoW/qrcwe7dqPTyqqnjIs+Mc8xOIKBJMdYJrAG0La4I1hv4stsfm/iMM+irhSfjSCI4k6S18t7vcoyuvcZr0xXkjf/6tgQkwPc7ieGr6gv4/RhE9e7kTNCOUZO7znq/+B1xIJWGRltfsY4jDIQGAi7S+4i6ELkuuHnu6+0V3T6YlDXq80Y4wKQd0/wy5rgGehIiwOCwcO7wTnj9qixWV9Dd3QBul/EBIOZSDUByu4RMVitVAnsHgG1XHx7Dbrzig+MEiBwPrE3XdOoGRopOJyRyldTqND46tA8zgGGrmgR6k8PjLYSNpUhvhg5DUwTamQt7uyrl/uYqgRuwqLTVRz8xUG5k/tGqwN74LD71vo4omghGpqqaYEqnLrDzaM1NYV2MRRL1dYT9JsL7swLHs/roeX/yyn0GlMwxSYIx7AVDTMDrxMzIpClADV2rjPV5rsctDAbh8GzLbEE+qG3SptxiGp0IvyZW2wuwFs5OZm1jHmd47dWLUwmISnmB10nbWOV5TK63/9MvH9762j2Sgvt/ogjpODuS+1Ht5xxmdMK30BxyM2ojqS5oF/xlBOK9rYX9BRoTMyQwEJ20qqkxWgwCC+yMtFLls7hNa3A4LkNvlyujGhdmR3qaih6tOhJLgfk0Jg/obXE9RjVzYqhlNCAEQ3YFR1mjXtzphy+S3MVd+9xTzOa+1etphU2Klk7SxHTbBx3JIlkxuRh26fkSlRHs9tncnHgagJEWGbiIPojNUxabp2Dg4qxPjenxFHNIIIB7XkANy6HIlkwxRdjKBj6mDewxC3ytHKqijAkPWtBIutkaQpfWZL7j/uBv2ltVmeajG7XxXxW7bE1HSss6bxbM6OGMaQPn9o6vRnvT7SjG2lkDt/MqkIwA/xnbRWuqInULS2XYgP+MO6oTmiJn4+yOrvC4UlNcFUUT4D9iL/FBWzdJvkb1TpZgexoRHreLULxJMVLf+Rkeg+13YsrVfyzGoUEulIziUJLHLIeUhF3bvKsUbFcEbvc6BQ/OTfOAr9ikeUcF1eQarVAPVA5+VD3VABY8dmKMnYsS1iCwjSv+7BEI+MusuAh48fGT6LfyEu5QOqnOsDGeX1yUe4Y75rYldtQecOToz0XzBo2+jjoSp32TArWJSJrTRsC1VlHOYs9ZOEM4cCqD01t7rRra+MhOvtY9sRrvQmXFjXeHG6m1lk1VGTYwOEqVjo+1h9pc79dwmxZxuqZkXsj5ePjppLf4us/AzzrHQVpB2VlyTOQvLKaoqkuQZ1AqoEVOxufT5ai7NzrmzpTAEXVQjcRJGzwAXU3T34iN/1bfJcR3y13LddA1ErXROaCkY13ZAN4YaB2F/uHxMzW22hp41vIIEJwvrWQZtuLBtywWeUC4rZ4/EwtjQsklFFx7CoeB576jvJ9GNJ5Pz9ZfiBopqYwDuUY54o50EcFRUpYqXwcM2l7+KoPxU++7hefM433nZSoPWi4dw9lCnTtIlnKauiNy90pWKxU/F8xJz+XFF+L3nTmM7H25e+3Kv7wVaAFaiFoPmy/HNC93WuWAUUQGRicLfXFR3Lot4LkX2Y7D0V1xOmoD6OsF/DhKgiFmCiLMlt3k8akOP1O9UAs5m58eBF0VBejbXF9bmI574gV2TSY5svQmmgDEx1mcOaUpcSrZposJ1qhOrshSBbScC24ZAGpgHPZbT/RxyIpclypg5thBxgBiOyxbmHT986nFMZE1E3azzwlpC2ZptS1Lkz9rGSYuSGMsoXFbRqCT0UpAJgZBj7aQt8Ni1hlWmwO5HUAidsXGbFHa7MYAklMsqzCEdJFukxo9oM5Gwj5+502jU3nMbwdORkjjbZqraPRSwGNcFcFvveM+WAbDYEDEb/IonXxDea3+pLrsoQ1KlCq7Trvjz08w93mypGwdFTHwsmc2V1ULFHqJmpKJRgnr+is7XWm+URXpAjSkweHB9YIxUSWrxoTG2vCd9rKodT4QnAnFrdq7ArizpxV0fkQYnvNHuQcaLTo2FDemIpBQxzX7vnoo1Y2bsKBt0LoGHGvQb4Z+CSeuMpUlZU3+3VTnpMiyQ/8Q9SwWaAkrOIGzj1EAPMWCRVgZZIkXQZG1qlcLv1szKEzWTCXXSofzxU3J8QGyFvY1OKtgP6GtkDTN3sSxxT3V0JruWadZGXvksIG4us9p6d5DYk3nzIO55sXXYoVLN0ZOCvpB0J3LlexEKpfjci3yz6hcq9hVqyHT9fKC1YxRhQyIWuca4zP++7//DU1iq/N2cI5ejN2IvtVsoRdTOlQ6ukNThr1GupFmy34HXwbYPaN7ZTaFU9UHm509f0CjS7feZWOL9IbzvpMPySV3gk1yzd37YDw+MKm1GmVeW1MxbqMGWHaaxxegwj9iOKJVWzqaU3H0BB1nLFbbgPbdoRYWgRmZe9ARYm2MOcYfTRch+d5i4fmr3Trx3TOi8/XQEdX7fHVVFTmWCkjzjKX960GCQLdqgmVJCOoQw3VDKTPP1DgOZLGxNEmPhfOop71HZVe7vBvR1DGaeRuY5CPSqRAJ7W051G6IHuqjJOXH8mAXBhYPdrLT1e9cwRpAU16rrNvbqSp0dNVc3iJOxE0HhqA5uDXFLXy5j21mHhZxCqd7W1DiGP1LuAEFDt0BPAeGV4GdJjQsEKgokMCP15TCsKGyKvmNva14rRiRPSnuzGHLoVDwQxhfWBh+WSVubNE0gn/QxefmzYg43BxNJxlX+hmmM0/oZq8G2NCjvqUmbDe+X6Ih8ikIgFWUlpC2C7fCdNG2tF3IENOFz2h0n8yXMnuhQJWkzd5AwW+ZdcF1mi5edNsJC1dVYnqRDaKPWWE7+adbzWnrDnSqIaAik2NlsZgWnB9gEHqLbLNhIQJM3JbbMt5xhy8HwJBF6Gm0wBfFen8EGPjZKVPdJmkedIoVqKIfXQ5T3R+9rC53+IbAO+oJpNmwxRx6xcUGi96A79Um2WVN/YPKyjcGWBwenipK1us40UMCfzLh1z4mp+vyBvPaeC1xAMe8eCHN6GEUzVWl1AQQTOhg/U4s+tKZtHH234sJ1fcEQ9t/CMEfp8NEB2qDIKV0Nu/T4mR6dDC/wDI48nQ6fRAnyfqboPU3iOb5s6NYwF+eoEqZYAFPwrlGfDa46PIVy4mm96Iz9Ws9vAdwTqOT+5E2KoGtqiZUQ3WEwLNjBLZiPZm0aV2gM0tX+wm4UCrzpUfBKH3UYgrkkVSkzcjGPEwMuAKZXPg/7LZJPsGXGtCzwEv2Gnx0jB/SBGD2FPiiExnk+AYRylV9VWRr46wc5QUbOr/3rIINZs/rAKrjRzVJs0mBK8N7GYYlK04y0oLiBuY2cdDqEu1LjYf+ICbMD2r9zfRSZNHNfSAYj1jHqKxCt8d42K3UdgDYpjvaO5ANw3bMfUn+m3QieHWYh2Z/yXh16NB1VqFDzpzCNkHTZSc52ZrQmjl0kwQ23mxfgyOcc88H7w447Ud4uQUaibaBtfm63aYU6bXvekW0/Jh3VwSw/MsUd82v1DVfB/jjh9cvv8cCj9XNeuGyCI4F2EJ0pHQ6GrOuZWCTenL+rxaeeFXqd9XD6NfyCN2dQNYWxogJe3lWkQ9YDKTgaGl0gcVwqKgtojDitTIuBbs81gXjnwF7Prhc4e9gteg1SAPKN1z1GIf2KNVu7EUKKfvf63DKHeP68mdgx418/xG6Whq/+BhR2dVXWoo5aogagyNa+C4d5/y7MjKWYabQPMSmmIMYIgq3DgmSJ2v6Tb94U+CuX7nTnepc9ywHanf6sNwjYW39iSjj6Y+zncvjhTyHRgqg5WApTwz7FHMhNKAJ+nhEWc4S89l3X8ZUduPWlLZYuBqxPbLHioHk2l2EfTr6VUKanHYqt/YpBvz9mqIvzrHVsUlRCQHnFF8rjda7bVkH3aMx7p5ct6SmUzzkKplWCMNh/e0cSid8ZFC7MaVD6LU8UT0H6mGO3+Ebdm6mLc3X6jZEQY1NDgh1vcp36O40KmCJhCOQwvkTMTYLvaCyJYnAiawETubycNJzfCA5blZgoiRE8FgmPe0aH6yFMRkm3wu3CU/Mp9aMEF/NvrJvi68Hk9E9pbzRt7axkDElOJ3LClxT8GOIHvUTe/y2IkeOEBIuQhcrh4gNBhFjA/szpdeaOZCZpTm9yO0uftoZjq9UULn3Qk4sYnmUbB0uRzarOTfYKHdpyBgv2X7QP/FcWQqXI5tB1pZu3IaSjsWacUQ373D/CK7tfQigJoZjUw8l5H5oMERjs4EaslMALOSSlxXzWwS1SB+QoLfNd74mF1QnxoI9U83Kv7+MXDHHDTCbdlCQHXk/p3EiBW2zZWQ36GCwGfKxnA2NwRD+EDi2D4DbfP/QGNM5MK7Nkg4NxNSIGYPaprWTjWGjRc2RFTehSJLS68dlHOiySznQb1dlkjd2VcfhKTx2ANZRvqQ32OX2/iI10dDKBXDYdrr47BBXsQ1XMdHq64ytTPobnAjMYOqfCN2FeSFZwdErxnHzSBz9NIdXVOKci3mXQ2qrozzM2zo4+fHEZgdD2C/mGMhdSWo6I+RF7nY5Jmub7XKBTL6yv8SuAvl6YdYo8nqu0rV+gEjydNIfXVHsHVpnrYeqKVxKb2NQoxxuj/kV8uE6EiuEXDYiS0UEsqFF0AwOZWPxAvR0LEtUW5uDSgN6uAyZj0SnkwCn/PpZp1Sgx1cLLRgrFZbuHyiJsCPlJxFC5jHXk1AlwLSDi+q7xRvtjz4JFotD8dEDYCnkdw77BLGJNPASsysnQy85Di1qaOoTfrOc3vFr5cK8P/lIHnSW/2BJsNRYtWjew+yQZHz938sVxnI+pXj/QW4ITmDgge7SO9d21p4OmBj6ydVKPn7Z4rbThoIDzfin06PPrD/vM1x3DbDPzSp3lvCik5OXL3wMzK1TTfb5HnCdYxJ7cnhAm5qzy9Mv5olbEHhuvzfQQ6MtuIdnu41a52AY6pt+3rebGV9074JDCi30njzhUzFsbfw7kujSwj235u3y4Un1h/HJ3K2P4lPXKvj/wydj9j+AT72jorWte3KMQovEAqkcocdBPdxl6P3DtQBBI87NHwroQGiCKGKM0fWjb+bL1YQCe5/gh+GTyxvGR4kZg4ffdO8MEiRJtaq/e/Am0rGyOt2CKqjSZh/0bdOubLofh+h6loMFPHe9ch7fwINeshZ0H0rn6uZCWQ0AaVtTBv7Mp424rf8lpTb+KVik94JTzzzeOQJH0Ei+D6Bxdv4IGt7JNn13LxIQq8eS6KyYTo0zC7U8Ag2n9ofRcG5/GNmXAc3E8VAR+JQn63xywgXJnYD9oSCOOYnd9rEzqqdR9ahuuztKB3PsFAPC2Q3ldMlxZbIbyumSwdAy5uiEc47FHTHUmBf5hL55xdFGGFvjlx3B0/xYezeqUvbTjWtfT3PQPcParyF+u1mMg8OGGdsxeu0czOfxaOi6GKSmZWvnehimogX/48re/RjgH1f2Lr7HKXtn6L9L4TtIjdattsRerS65lvJT1QRvIuiIM/y20P1TjMfuJqDGaJEOKTiHxw74oK7V+UarWIW9QIMnztTON4Vid5j89ZTefZAjQ/zi3AuxYy5tjnIloEv9UoesrLqgwIKKu7cBKFc5+3cLjtIADfHQxSHrr8xaBlAOrdOOlHMcROKkHg9dMkh8yxCJ18DaS0SmyEwaDSu5siwgdkX8Vca2MpVeT3V4WAPGbRJfg6LjVZ4MfLEg3tA1jaZw3H5pFq0NDSDG0MTQRX9l/rQtJZl3/VSR++TClZjrXea89gPdPcd1eHcNlvt336I5dgoes8sW4b0mxX17TPssj8+qoPrHDiPxewq1wiPkdz9rTF9q1rXcX7ffXKbSBp1gw5cTOEjid9bA92DM9yBgxwi3vFe77rQ1RxFQK+kuSJaWsb5MmbOd1x94ndsyi/WnZw6AbBW+cR/DVa96aWkCeN/5VHJxk/PXXcVXk//c+2iy30f07uWEP6XcflC5XUDUeflueN/aiqv+5okaEugUv8IhuPbjCvzlWywqEN/NHcpYd7egX2hxBFcf+BA6UwMwUObQPVZtociRmWU5yTBX9e3YZ6ktlGjrY/VH5DqU4OWMJrm4q4enMvJJrL2uY/68eEyfF8c7ols67NzJ44EL4KGInNt6CNE2zXd13GYZO6xwHa6OgjysvVwt1xt/4FoMB+Y2l1tvZdRxYGcvy12MgUsTP+mvq2MzxTovxLR1sitDgtS1sIzo9QNbQ6O7FltvtA33HNAHQonyt4jtT+dyRzngeA2KAyCMth+xAoN/8FuCoafATm7i4qModpEjb8COVTGWzknXka2CkPLOebOYjbGu77+wIJZeksCXefxds5m8MCVkgsZYf8qD5VR26GgdWPQOBWlzBfsOl89t4Eer+tqX3+vuItYf78aq8FzdZCC2C3+IrKEve9Na0Z2ASaLv01XzKzUEDAfecaqyNdWQLDDD7zrQ06WIWzMmZh5+ucwJTsrOqrhxHXHjVGxLfvni7o9ZVo+4hO+5gA9Egvgk2HShvzw3ChSeCGY5oAl7kRzHzB/dH/mR/kU4+gMK6w8rq/tDP61lPjDqcaWHLszjygplVZwQZX3UhCy7VZ/vaYPn+B03K5NfOpWh70QZtse2FRaD9oTzi+8WUmEJNpbwcQk2f8kETCpDeTfy8ssejNPt69u0W5n4/vU/Xr/68Pp7+06y3mv9/5UhrDcyXc1tImw5Weo1GqX4rU4U9DimCH4c41swcayj+PxKzOh/AYgUi6TmZAAA'),
}
for name, (expected, payload) in embedded.items():
    contents = gzip.decompress(base64.b64decode(payload))
    actual = hashlib.sha256(contents).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Embedded source checksum mismatch: {name}')
    (SCRATCH_DIR / name).write_bytes(contents)
print({name: digest for name, (digest, _) in embedded.items()})


In [ ]:
# Fetch exact source commits and frozen public weights, with visible progress.
import shutil, subprocess, time, urllib.request, zipfile

WHAM_COMMIT = '2b54f7797391c94876848b905ed875b154c4a295'
HMR2S_COMMIT = 'd69218f411e003621f29df1940b23b076067fad1'
HMR2S_CHECKPOINT_SHA256 = '823728e846c901c75edb12d469fa240e07606a24cfd44c208244a94bb26fc423'

def checkout_exact(url, commit, destination):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    subprocess.run(['git', 'init', '-q'], cwd=destination, check=True)
    subprocess.run(['git', 'remote', 'add', 'origin', url], cwd=destination, check=True)
    print(f'Fetching {url} at {commit}...', flush=True)
    subprocess.run(['git', 'fetch', '--depth=1', 'origin', commit], cwd=destination, check=True)
    subprocess.run(['git', 'checkout', '-q', '--detach', 'FETCH_HEAD'], cwd=destination, check=True)
    actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=destination, text=True).strip()
    if actual != commit:
        raise RuntimeError(f'Expected commit {commit}, got {actual}')
    return actual

def download_with_progress(url, destination):
    last_print = [0.0]
    def hook(blocks, block_size, total):
        now = time.monotonic()
        if now - last_print[0] >= 15 or (total > 0 and blocks * block_size >= total):
            received = blocks * block_size
            total_text = 'unknown' if total <= 0 else f'{total / 2**30:.2f} GiB'
            print(f'  {destination.name}: {received / 2**30:.2f} GiB / {total_text}', flush=True)
            last_print[0] = now
    urllib.request.urlretrieve(url, destination, reporthook=hook)

WHAM_REPO = SCRATCH_DIR / 'WHAM'
HMR2S_REPO = SCRATCH_DIR / 'TruncHierVFM'
checkout_exact('https://github.com/yohanshin/WHAM.git', WHAM_COMMIT, WHAM_REPO)
checkout_exact('https://github.com/nttcom/TruncHierVFM.git', HMR2S_COMMIT, HMR2S_REPO)

downloads = {
    'wham_vit_bedlam_w_3dpw.pth.tar': (
        'https://huggingface.co/camenduru/WHAM/resolve/main/'
        'wham_vit_bedlam_w_3dpw.pth.tar?download=true',
        '2ba0cb6a7dd597023a6b2ad6056e7a8b6b33144a35fabea570bfd00842cd4eaf',
    ),
    'yolo26n-pose.pt': (
        'https://github.com/ultralytics/assets/releases/download/v8.4.0/'
        'yolo26n-pose.pt',
        'eb3bb8268828aeaf515cec23a4bfafd793944a86fe9af94ba7823609c14522a9',
    ),
    'J_regressor_h36m.npy': (
        'https://huggingface.co/camenduru/WHAM/resolve/main/'
        'J_regressor_h36m.npy?download=true',
        'c655cd7013d7829eb9acbebf0e43f952a3fa0305a53c35880e39192bfb6444a0',
    ),
    'J_regressor_wham.npy': (
        'https://huggingface.co/camenduru/WHAM/resolve/main/'
        'J_regressor_wham.npy?download=true',
        'f938dcfd5cd88d0b19ee34e442d49f1dc370d3d8c4f5aef57a93d0cf2e267c4c',
    ),
}
downloaded = {}
for name, (url, expected) in downloads.items():
    path = SCRATCH_DIR / name
    if not path.is_file():
        print(f'Downloading {name}...', flush=True)
        download_with_progress(url, path)
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f'{name} checksum mismatch: {actual}')
    downloaded[name] = path
    print(f'Verified {name}: {actual}', flush=True)

hmr_files = []
for parent, directories, files in os.walk(KAGGLE_INPUT):
    directories[:] = [d for d in directories if d not in {'imageFiles', 'sequenceFiles'}]
    for filename in files:
        if filename == 'last.ckpt' or filename == 'hmr_vit-small_d3-a4x16-m128.zip':
            hmr_files.append(Path(parent) / filename)
checkpoint_matches = [path for path in hmr_files if path.name == 'last.ckpt' and sha256_file(path) == HMR2S_CHECKPOINT_SHA256]
if checkpoint_matches:
    HMR2S_CHECKPOINT = checkpoint_matches[0]
else:
    archives = [path for path in hmr_files if path.name.endswith('.zip')]
    if len(archives) != 1:
        raise FileNotFoundError(
            'Attach one Kaggle input containing the authors’ official '
            f'hmr_vit-small_d3-a4x16-m128.zip; found {hmr_files}'
        )
    hmr_extract = SCRATCH_DIR / 'hmr2s_weights'
    print(f'Extracting the attached HMR2.0-S archive: {archives[0]}', flush=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(hmr_extract)
    candidates = list(hmr_extract.rglob('last.ckpt'))
    checkpoint_matches = [path for path in candidates if sha256_file(path) == HMR2S_CHECKPOINT_SHA256]
    if len(checkpoint_matches) != 1:
        raise RuntimeError(f'Attached archive is not the pinned official HMR2.0-S release: {candidates}')
    HMR2S_CHECKPOINT = checkpoint_matches[0]
actual_hmr = sha256_file(HMR2S_CHECKPOINT)
if actual_hmr != HMR2S_CHECKPOINT_SHA256:
    raise RuntimeError(f'HMR2.0-S checkpoint checksum mismatch: {actual_hmr}')
WHAM_CHECKPOINT = downloaded['wham_vit_bedlam_w_3dpw.pth.tar']
YOLO26_WEIGHTS = downloaded['yolo26n-pose.pt']
H36M_REGRESSOR = downloaded['J_regressor_h36m.npy']
WHAM_REGRESSOR = downloaded['J_regressor_wham.npy']
print('All frozen public artifacts verified.', flush=True)


In [ ]:
# Locate the three private licensed SMPL files without scanning imageFiles.
SMPL_MODEL_DIR = SCRATCH_DIR / 'licensed_smpl'
SMPL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
aliases = {
    'SMPL_NEUTRAL.pkl': {'SMPL_NEUTRAL.pkl', 'basicModel_neutral_lbs_10_207_0_v1.0.0.pkl'},
    'SMPL_MALE.pkl': {'SMPL_MALE.pkl', 'basicmodel_m_lbs_10_207_0_v1.0.0.pkl', 'basicModel_m_lbs_10_207_0_v1.0.0.pkl'},
    'SMPL_FEMALE.pkl': {'SMPL_FEMALE.pkl', 'basicModel_f_lbs_10_207_0_v1.0.0.pkl'},
}
wanted = set().union(*aliases.values())
found = {}
for parent, directories, files in os.walk(KAGGLE_INPUT):
    directories[:] = [d for d in directories if d not in {'imageFiles', 'sequenceFiles'}]
    for filename in files:
        if filename in wanted:
            found.setdefault(filename, []).append(Path(parent) / filename)
for destination, names in aliases.items():
    matches = sorted({path for name in names for path in found.get(name, [])})
    if not matches:
        raise FileNotFoundError(
            f'Missing licensed {destination}. Attach your private SMPL model dataset.'
        )
    shutil.copy2(matches[0], SMPL_MODEL_DIR / destination)
    print(f'{destination}: {matches[0]}')


In [ ]:
# Fast-fail smoke test before the full evaluation.
import json, subprocess, sys

environment = os.environ.copy()
environment['PYTHONPATH'] = str(SCRATCH_DIR) + os.pathsep + environment.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-m', 'py_compile', *[str(SCRATCH_DIR / name) for name in embedded]], check=True, env=environment)
smoke = f'''
from pathlib import Path
import torch
from hmr2s_frozen import FrozenHMR2S
model = FrozenHMR2S(Path({str(HMR2S_REPO)!r}), Path({str(HMR2S_CHECKPOINT)!r})).eval()
with torch.inference_mode():
    outputs = model(torch.zeros(1, 3, 256, 256))
expected = [(1, 1024), (1, 24, 6), (1, 10), (1, 3)]
actual = [tuple(value.shape) for value in outputs]
assert actual == expected, (actual, expected)
assert all(torch.isfinite(value).all() for value in outputs)
print({{'hmr2s_shapes': actual, 'parameters': sum(p.numel() for p in model.parameters()), 'training': False}})
'''
subprocess.run([sys.executable, '-u', '-c', smoke], check=True, env=environment)
print('Smoke test passed; starting the locked comparison next.', flush=True)


In [ ]:
# Full two-row comparison. Progress and one compact line per track are printed.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT = OUTPUT_DIR / 'frozen_hmr2s_wham_accuracy_3dpw.json'
PER_SEQUENCE = OUTPUT_DIR / 'frozen_hmr2s_wham_accuracy_3dpw.csv'
EVALUATOR = SCRATCH_DIR / 'evaluate_frozen_hmr2s_wham.py'
command = [
    sys.executable, '-u', str(EVALUATOR),
    '--parsed-3dpw', str(PARSED_3DPW),
    '--three-dpw-root', str(THREEDPW_ROOT),
    '--wham-repo', str(WHAM_REPO),
    '--wham-checkpoint', str(WHAM_CHECKPOINT),
    '--hmr2s-repo', str(HMR2S_REPO),
    '--hmr2s-checkpoint', str(HMR2S_CHECKPOINT),
    '--yolo26-weights', str(YOLO26_WEIGHTS),
    '--smpl-model-directory', str(SMPL_MODEL_DIR),
    '--h36m-joint-regressor', str(H36M_REGRESSOR),
    '--wham-joint-regressor', str(WHAM_REGRESSOR),
    '--sequences', str(SEQUENCES),
    '--frames', str(FRAMES_PER_SEQUENCE),
    '--pose-batch-size', str(POSE_BATCH_SIZE),
    '--hmr-batch-size', str(HMR2S_BATCH_SIZE),
    '--smpl-batch-size', str(SMPL_BATCH_SIZE),
    '--output', str(REPORT),
    '--per-sequence-output', str(PER_SEQUENCE),
]
print('Launching frozen evaluation (training=False)...', flush=True)
subprocess.run(command, check=True, env=environment)


In [ ]:
# Show the final numbers and create one small download bundle.
import pandas as pd
from IPython.display import display

report = json.loads(REPORT.read_text())
table = []
for variant, result in report['variants'].items():
    metrics = result['metrics']
    table.append({
        'pipeline': variant,
        'frames': metrics['mpjpe_mm']['samples'],
        'PA-MPJPE (mm)': metrics['pa_mpjpe_mm']['mean'],
        'MPJPE (mm)': metrics['mpjpe_mm']['mean'],
        'PVE (mm)': metrics['pve_mm']['mean'],
        'Accel (m/s²)': metrics['accel_official_30fps']['mean'],
    })
display(pd.DataFrame(table).round(3))
print('Phone minus released WHAM:', json.dumps(report['phone_minus_released_same_population'], indent=2))
print('Relative change:', json.dumps(report['phone_relative_change_vs_released_same_population'], indent=2))
print('Detection:', json.dumps(report['detection'], indent=2))
MANIFEST = OUTPUT_DIR / 'artifact_manifest.json'
manifest = {
    'training_performed': False,
    'report': REPORT.name,
    'per_sequence': PER_SEQUENCE.name,
    'send_back': [REPORT.name, PER_SEQUENCE.name],
}
MANIFEST.write_text(json.dumps(manifest, indent=2) + '\n')
bundle_base = Path('/kaggle/working/frozen_hmr2s_wham_accuracy_results')
bundle = Path(shutil.make_archive(str(bundle_base), 'zip', root_dir=OUTPUT_DIR))
print(f'Download {bundle} ({bundle.stat().st_size / 1024:.1f} KiB).')
print('Send back the JSON and CSV, or just this ZIP. No logs/checkpoints are needed.')
